# Spatial–Temporal Feature Fusion with Multi-Head Attention
## UCF-Crime2Local: repository and experimental workflow

Self-contained source notebook for GitHub and Zenodo. It materializes the same
Python modules supplied separately in the repository archive. Run cells in order.
Use the actual dataset, exact eligible partition manifests and annotation records.

**Architecture:** custom YOLOv8n + R3D-18 convolutional fusion with residual spatial
attention and dense key-frame heads; not the official YOWOv3 implementation.
**Protocol:** 30 epochs, AdamW, batch eight, accumulation two, validation-loss
checkpoint selection; no NMS in primary evaluation; confidence 0.3, AP floor
0.001, IoU 0.5 and fixed six-class all-point AP.

Author-confirmed manuscript values are preserved in `reported/reference_values.json`
for comparison. This notebook does not manufacture checkpoints or prediction
records that produce those values. The supplied notebook contained no executed
research outputs. Dataset bytes and historical evidence must be supplied or
produced by actual execution. Dataset/weight terms remain applicable.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
WORK = Path(os.environ.get("UCF_WORK_DIR", "ucfcrime2local_experiment")).expanduser().resolve()
WORK.mkdir(parents=True, exist_ok=True)
print("Source and run workspace:", WORK)


## 1. Materialize the refined source
Contains training/evaluation, dataset auditing, analyses, profiling, real overlays,
tests, export utilities and documentation. Generated run artifacts are separate.


In [ ]:
BUNDLED_FILES = {'.github/workflows/protocol-tests.yml': "name: Protocol tests\non: [push, pull_request]\njobs:\n  tests:\n    runs-on: ubuntu-latest\n    steps:\n      - uses: actions/checkout@v4\n      - uses: actions/setup-python@v5\n        with:\n          python-version: '3.10'\n      - run: python -m pip install numpy==1.26.4 pandas==2.2.2 PyYAML==6.0.2\n      - run: python -m compileall -q .\n      - run: python -m unittest discover -s tests -v\n", '.gitignore': '__pycache__/\n*.pyc\n.ipynb_checkpoints/\n.venv/\nruns/\nrelease/\n*.pt\n*.pth\n*.zip\nucfcrime2local/frames/\nucfcrime2local/videos/\nucfcrime2local/annotations/*.csv\nconfigs/local.yaml\n', 'CHANGELOG.md': '# Refinement summary\n\n- Align standalone and notebook defaults: six classes, 16 frames, 224×224, 30 epochs, seed 42, workers two, no primary NMS, AP floor 0.001.\n- Replace automatic CPU fallback with explicit device selection.\n- Preserve routed YOLO connections and initialize lazy projection before optimizer construction.\n- Add optional controlled components and 8/32-frame retraining with isolated run histories.\n- Export one-based epochs, per-epoch learning rate, actual counts/AP and checkpoint-linked predictions.\n- Correct assignment comments: one regression target per box-center cell, not FCOS center sampling.\n- Add validated CSV adaptation, original-video frame extraction, split audit, collision counts and intermediate shape traces.\n- Add threshold/NMS/error/cluster-bootstrap analyses, original-pixel overlays and record-derived chart generation.\n- Record profiling scopes, partial THOP coverage and actual stage timings; distinguish JPEG frames from video containers.\n- Add author-reported reference manifest, tests, GitHub CI, citation, attribution and release/export documentation.\n- Notebook has no manufactured research outputs or placeholder checkpoints.\n\nSeven protocol tests passed; the optional dependency-backed model test was skipped in the preparation environment. See VALIDATION.json for scope.\n', 'CITATION.cff': 'cff-version: 1.2.0\nmessage: "Please cite this software and the associated manuscript."\ntype: software\ntitle: "Spatial–Temporal Feature Fusion with Multi-Head Attention for Supervised Anomaly Localization"\nversion: 1.0.0\nauthors:\n  - family-names: Ramzan\n    given-names: Saba\n  - family-names: Khan\n    given-names: Sajid Ullah\n  - family-names: Arslan\n    given-names: Farrukh\n  - family-names: Jaffar\n    given-names: Arfan\n  - family-names: Naeem\n    given-names: Muhammad\n  - family-names: Iqbal\n    given-names: Muhammad Javaid\nabstract: "Custom YOLOv8n and R3D-18 feature fusion with residual spatial attention for supervised key-frame anomaly localization, with reproducible experimental workflows."\n', 'DATASET.md': '# UCF-Crime2Local dataset preparation\n\nThis package expects consecutive source frames and actual spatial annotations,\nnot image-level anomaly labels. User-supplied raw dataset bytes are separate from\nthis source release. Keep original dataset attribution, license/terms and source\nchecksums with the dataset archive.\n\n## CSV contract\n\n`video_id,frame_idx,cls,x,y,w,h,t`\n\n- `video_id`: path relative to `frames/`, e.g. `Arrest/Arrest028`; the same spelling\n  is used throughout annotations, decoded frames and predictions.\n- `frame_idx`: integer, zero based. Frame zero is the first decoded source frame.\n- `cls`: 0 Arrest; 1 Assault; 2 Burglary; 3 Robbery; 4 Stealing; 5 Vandalism.\n- `x,y,w,h`: normalized center coordinates and dimensions relative to the original\n  frame. Positive widths/heights, finite values, whole GT box within image bounds.\n- `t`: binary temporal metadata, retained but not used as a localization target.\n- One row per actual annotated box. An explicitly verified negative frame has\n  one row with `cls=-1`, `x=y=w=h=t=0`. Missing annotations are not negatives.\n\nThe three eligible manifests must be video-disjoint. Dataset audits detect\nidentifier overlap, malformed boxes, duplicate rows and missing causal context\nframes. They do not establish absence of renamed duplicate video content.\n\n## Explicit adaptation\n\nIf your native data is tabular, `import_annotations.py` converts a CSV with\nexplicit options. It does not guess the raw UCF-Crime2Local annotation format.\n\n```bash\npython import_annotations.py --input original_train.csv --output ucfcrime2local/annotations/train.csv --frames-root ucfcrime2local/frames --coordinates normalized-xywh --frame-base 0\n```\n\nFor pixel corner boxes use `--coordinates pixel-xyxy` with columns\n`xmin,ymin,xmax,ymax,image_width,image_height`. If input columns differ, pass a\nJSON mapping canonical names to existing names with `--mapping`. Use\n`--frame-base 1` only when source indices are explicitly one based. Do not invent\nclass IDs, boxes, t values, negative rows, or historical split membership.\n\nIf your archive contains original videos, provide a JSON mapping each video_id\nto its original video file and decode the required context windows:\n\n```bash\npython extract_frames.py --video-map video_map.json --annotations ucfcrime2local/annotations/train.csv ucfcrime2local/annotations/val.csv ucfcrime2local/annotations/test.csv --frames-root ucfcrime2local/frames --clip-length 32\n```\n\nExtraction uses sequential zero-based OpenCV decoding and JPEG quality 95. Match\nthese settings to the actual historical extraction before claiming exact\nreproduction. Existing frames are protected unless `--overwrite` is supplied.\n32-frame extraction supports all context-sensitivity runs. Variable-frame-rate\nvideo correspondence must be resolved against the original annotation convention.\n\n## Counts and target collisions\n\nAuthor-reported eligible partitions: 52/13/31 source videos, 20,000/5,000/8,000\nkey frames, 25,000/6,250/10,000 box instances. Listed annotation-path counts\n31,687/13,226 refer to a different accounting unit and must not be reconstructed\nby arbitrary sampling. Preserve original inclusion/exclusion manifests.\n\nUse the recorded output grid from `tensor_shapes.json` with\n`audit_dataset.py --grid HEIGHT WIDTH` to count additional instances assigned to\noccupied regression cells. Do not assume a 7×7 grid without the trace. These\ntraining-assignment collisions do not remove GT boxes from evaluation.\n', 'README.md': "# Spatial–Temporal Feature Fusion with Multi-Head Attention for Supervised Anomaly Localization\n\nCustom YOLOv8n appearance + R3D-18 temporal architecture with convolutional fusion,\nresidual spatial attention and dense key-frame localization heads. This is the\nmanuscript implementation, not the official YOWOv3 baseline. The six classes are\nArrest, Assault, Burglary, Robbery, Stealing and Vandalism.\n\n## What this release contains\n\nExecutable training, evaluation, dataset auditing, controlled component variants,\nthree-seed runs, 8/16/32-frame retraining, fixed-checkpoint threshold sweeps,\nclass-aware NMS sensitivity, false-positive attribution, video-cluster bootstrap,\nshape traces, forward/offline profiling, charts, record-derived overlays, an\nupdated self-contained notebook and checksum/export utilities.\n\n`reported/reference_values.json` preserves the author's confirmed manuscript\nvalues. It is a comparison reference. Generated `runs/` outputs are produced only\nby execution. This source release contains no historical checkpoints, prediction\nrecords, training histories, GPU measurements or research dataset bytes, because\nthose records were not contained in the supplied notebook. Unit-test fixtures are\nsynthetic software checks and are not manuscript results.\n\n## Installation\n\nUse Python 3.10 (manuscript version 3.10.13) and a suitable NVIDIA driver. Create a\nvirtual environment, then install the manuscript CUDA build and pinned packages:\n\n```bash\npython -m pip install torch==2.2.2 torchvision==0.17.2 --index-url https://download.pytorch.org/whl/cu121\npython -m pip install -r requirements.txt\npython -m unittest discover -s tests -v\n```\n\nThe CUDA 12.1 command is from the official previous-version installation guide:\nhttps://docs.pytorch.org/get-started/previous-versions/\nAdditional package pins provide a reproducible proposed environment; their exact\nhistorical versions were not all specified in the manuscript. Every execution\nrecords installed versions. Download access is needed for the pretrained YOLOv8n\nand Torchvision R3D-18 weights on first use; preserve their cache and provenance.\n\n## Add `ucfcrime2local`\n\nExtract the dataset into `ucfcrime2local/` beside the repository, or use an external\nabsolute path. Required layout:\n\n```text\nucfcrime2local/\n  frames/<video_id>/00000.jpg\n  frames/<video_id>/00001.jpg\n  annotations/train.csv\n  annotations/val.csv\n  annotations/test.csv\n```\n\nRead `DATASET.md` before adapting native records. Original annotations are not\nassumed to already satisfy this CSV contract. Original video splits, frame-index\nconventions, box coordinate conventions, and negative-frame verification must\ncome from actual records. Do not infer negative examples from missing boxes.\nSupply the exact eligible-key-frame manifests used for the reported split.\nThe loader does not automatically turn 31,687 listed paths into 20,000 training\nkey frames or choose a historical validation partition.\n\n```bash\npython configure.py --data-root /absolute/path/ucfcrime2local --output configs/local.yaml\npython audit_dataset.py --config configs/local.yaml --output artifacts/dataset_audit.json\n```\n\nCUDA is required by default. For an explicit CPU smoke test use `--device cpu`\nwith `configure.py`; CPU timing does not represent manuscript GPU performance.\nAll commands are executed from the repository root.\n\n## Reference run\n\n```bash\npython train.py --config configs/local.yaml --out_dir runs/complete_T16_seed42\npython evaluate.py --config configs/local.yaml --checkpoint runs/complete_T16_seed42/best.pt --output runs/complete_T16_seed42/evaluation\npython analysis.py --predictions runs/complete_T16_seed42/evaluation/predictions.json --output runs/complete_T16_seed42/analysis --bootstrap 1000\npython profile.py --config configs/local.yaml --checkpoint runs/complete_T16_seed42/best.pt --output runs/complete_T16_seed42/profiling\npython plot_results.py --runs runs --output runs/figures\npython export_overlays.py --predictions runs/complete_T16_seed42/evaluation/predictions.json --frames-root /absolute/path/ucfcrime2local/frames --output runs/figures/Figure8.PNG\npython check_reported_values.py --results runs/complete_T16_seed42/evaluation/results.json\n```\n\nUse the workflow driver below when raw console logs are needed; it captures every\nsubprocess automatically. The standalone commands print to the terminal.\n\n## Full manuscript experiment matrix\n\n```bash\npython run_experiments.py --data-root /absolute/path/ucfcrime2local --all\n```\n\nThis performs eight 30-epoch training runs: the reference (seed 42), seeds 21/84,\nspatial-only, temporal-only, fusion without attention, and independently retrained\n8/32-frame complete models. Each has its own config, best/last checkpoints,\nhistory, logs, environment, predictions and results. The reference additionally\nreceives threshold/NMS/error/bootstrap analyses and profiling. No significance\nclaim is inferred from component differences. Seeds do not make CUDA kernels\nuniversally bitwise deterministic; the software/hardware environment matters.\n\nRun directories are protected against accidental training overwrite. To evaluate\nexisting populated directories use `--skip-training`. This still evaluates and\nrewrites derived summaries; preserve historical files before rerunning analysis.\n\n## Protocol\n\n- 16 causal RGB frames ending at the annotated key frame; repeat frame zero near\n  the beginning; resize to 224×224; shared `(RGB/255 - 0.45)/0.225` normalization.\n- YOLOv8n feature routing; R3D-18 convolutional stages; 256-channel fusion; four\n  attention heads of dimension 64; dense class/objectness/normalized box outputs.\n- AdamW, batch eight, accumulation two, 30 epochs; learning rates 1e-4, 5e-5,\n  2.5e-5 and 1.25e-5 over epochs 1–10, 11–18, 19–24 and 25–30.\n- Minimum mean validation mini-batch loss selects `best.pt`; test data never\n  select the checkpoint. The checkpoint `epoch` field is one-based in this release.\n- P/R/F1 use confidence 0.3; AP uses floor 0.001, all-point interpolation and a\n  fixed six-class denominator (absent classes get zero AP).\n- Highest-IoU same-class greedy matching at IoU 0.5. If that target is occupied,\n  reject the prediction without reassignment. Primary evaluation has no NMS.\n- One regression target per cell; later boxes overwrite coordinates on collisions.\n  Multiple class labels may remain active at that cell, matching the supplied code.\n- Data-loader workers are two, matching the supplied notebook's override.\n\n## Timing boundaries\n\nForward-only timing: FP32 batch one, resident synthetic device tensors, ten\nwarm-up iterations, 100 synchronized individual measurements; one predicted key\nframe per clip. THOP reports accounted operations with two FLOPs per MAC; custom\nattention matrix operations are incompletely counted.\n\nOffline timing in this release reads/decompresses pre-extracted JPEG frames,\nassembles clips, preprocesses them, transfers inputs, performs a forward pass,\nand decodes/serializes predictions in memory. It does **not** measure decoding\noriginal compressed-video containers, writing outputs to disk, camera acquisition,\nframe waiting, networking, display, queueing or asynchronous overlap. Do not\nrename this JPEG-frame measurement as video-container decoding. Training peak\nallocated memory uses real clips, batch eight and two accumulation batches, with\nan AdamW update. Allocated memory and reserved memory are different quantities.\n\n## Export and deposit\n\n```bash\npython export_release.py --repository . --runs runs --output release\n```\n\nUpload source files/notebook to GitHub. Deposit source and actual experimental\nrecords (including real checkpoints and predictions) on Zenodo. Upload the dataset\nseparately with its own original attribution and applicable redistribution terms.\n`RELEASE_CHECKLIST.md` documents final metadata and evidence checks. Dataset rights\nare separate from code licensing; this release does not assign a new license to\nUCF-Crime2Local. The repository does not publish itself or create a DOI.\n\n## Citation and attribution\n\nSee `CITATION.cff`, `THIRD_PARTY_NOTICES.md` and the manuscript. Fill in real\nrepository and DOI identifiers after the deposit is created. No DOI is invented.\n", 'RELEASE_CHECKLIST.md': '# GitHub and Zenodo deposit\n\n1. Extract the source archive and run the tests in the documented environment.\n2. Add the actual dataset, original source/terms, frame-index and coordinate\n   descriptions, exact partition manifests and extraction provenance.\n3. Run the notebook/workflow or deposit your historical run records. Keep\n   imported historical records identifiable and preserve their original hashes.\n4. Confirm that actual predictions reproduce the manuscript ledger and class AP;\n   use `check_reported_values.py` without overwriting generated results.\n5. Retain all component, seed, clip-length, NMS, error-analysis and bootstrap\n   records. Three training seeds and test-video resampling answer different questions.\n6. Verify the hardware/precision/timing scopes. The JPEG-frame profiler does not\n   prove a compressed-video-container decoding measurement.\n7. Add your actual code license, repository URL, version and DOI. Retain dependency\n   license notices. Dataset licensing is separate.\n8. Upload source/notebook/docs/tests to GitHub. Use Zenodo for large actual\n   checkpoints, prediction archives and the separately licensed dataset. Update\n   `CITATION.cff`, README and manuscript availability text with the resulting DOI.\n9. Preserve `source_sha256.json` and `run_sha256.json`. Check archive integrity\n   before publication. No external account publishing is performed by this package.\n\nThe source can be uploaded as a software deposit. A manuscript evidence deposit\nis complete only when its real experiment/data records are attached; source code\nalone does not supply those historical measurements.\n', 'THIRD_PARTY_NOTICES.md': "# Third-party attribution and licensing\n\nUltralytics YOLO: https://github.com/ultralytics/ultralytics\nLicensing: https://www.ultralytics.com/license\nThe upstream project offers AGPL-3.0 and enterprise licensing. Retain upstream\nnotices and select a compatible public release license according to the rights\nheld by the authors. This package does not assert ownership of upstream code or\nassign a license to pretrained weights or dataset bytes.\n\nTorchvision R3D-18: https://pytorch.org/vision/stable/models/generated/torchvision.models.video.r3d_18.html\nPyTorch/Torchvision licensing follows the upstream projects.\n\nUCF-Crime2Local: Landi, F.; Snoek, C. G. M.; Cucchiara, R. Anomaly Locality in\nVideo Surveillance. arXiv:1901.10364, 2019.\nThe original provider's terms apply independently to dataset redistribution.\n\nThe authors should add the final code license and license text before a public\nrelease. The repository preparation does not invent an approval or license grant.\n", 'VALIDATION.json': '{\n  "protocol_tests": {\n    "passed": 7,\n    "failed": 0,\n    "skipped": 1,\n    "skip_reason": "PyTorch/torchvision/Ultralytics unavailable in preparation environment"\n  },\n  "python_syntax": "all source files compiled successfully",\n  "analysis_cli": "passed using isolated synthetic software fixtures",\n  "chart_export_cli": {\n    "passed": true,\n    "charts_exported": 10,\n    "fixture_scope": "software test only; fixtures not included as research records"\n  },\n  "model_gpu_training": "not executed: GPU framework dependencies and UCF-Crime2Local dataset unavailable",\n  "historical_records": "not included in uploaded notebook; source release cannot reconstruct original checkpoints or predictions",\n  "reference_values": "author-confirmed manuscript comparison values retained; not substituted for execution outputs"\n}', 'analysis.py': '"""Record-derived localization analysis. Pure NumPy; no model or dataset image dependency."""\nimport argparse\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\nimport numpy as np\nfrom runtime import write_json, sha256\n\nNAMES=[\'Arrest\',\'Assault\',\'Burglary\',\'Robbery\',\'Stealing\',\'Vandalism\']\n\n\ndef iou(box, boxes):\n    boxes=np.asarray(boxes,dtype=float).reshape(-1,4)\n    if not len(boxes): return np.empty(0)\n    box=np.asarray(box,dtype=float)\n    left=box[:2]-box[2:]/2;right=box[:2]+box[2:]/2\n    lo=boxes[:,:2]-boxes[:,2:]/2;hi=boxes[:,:2]+boxes[:,2:]/2\n    overlap=np.maximum(0,np.minimum(right,hi)-np.maximum(left,lo)).prod(1)\n    return overlap/np.maximum(box[2:].prod()+boxes[:,2:].prod(1)-overlap,1e-7)\n\n\ndef match(detections, ground_truth, threshold=.5):\n    """Highest-IoU same-class target; no reassignment when that target is occupied."""\n    gt=np.asarray(ground_truth,dtype=float).reshape(-1,5); used=set(); rows=[]\n    for index,det in sorted(enumerate(detections),key=lambda v:-v[1][\'score\']):\n        overlaps=iou(det[\'box\'],gt[:,1:]); same=np.where(gt[:,0]==det[\'cls\'])[0]\n        best=int(same[np.argmax(overlaps[same])]) if len(same) else None\n        value=float(overlaps[best]) if best is not None else 0.\n        tp=best is not None and value>=threshold and best not in used\n        if tp: used.add(best);category=\'true_positive\'\n        elif best is not None and value>=threshold and best in used: category=\'duplicate_or_assignment_conflict\'\n        elif any(overlaps[k]>=threshold for k in range(len(gt)) if gt[k,0]!=det[\'cls\']): category=\'wrong_class\'\n        elif len(overlaps) and overlaps.max()>=.1: category=\'insufficient_overlap\'\n        else: category=\'background_or_negligible_overlap\'\n        rows.append({\'detection_index\':index,\'cls\':int(det[\'cls\']),\'score\':float(det[\'score\']),\n                     \'tp\':int(tp),\'fp\':int(not tp),\'target_index\':best,\'same_class_iou\':value,\n                     \'max_any_iou\':float(overlaps.max()) if len(overlaps) else 0.,\'category\':category})\n    return rows,sorted(set(range(len(gt)))-used)\n\n\ndef interpolated_ap(scored, gt_count):\n    if gt_count==0 or not len(scored):return 0.\n    arr=np.asarray(scored,dtype=float).reshape(-1,3)\n    arr=arr[np.argsort(-arr[:,0],kind=\'stable\')]\n    tp=np.cumsum(arr[:,1]); fp=np.cumsum(arr[:,2]);rec=tp/gt_count;prec=tp/np.maximum(tp+fp,1e-9)\n    rec=np.r_[0.,rec,1.];prec=np.r_[1.,prec,0.]\n    prec=np.maximum.accumulate(prec[::-1])[::-1];j=np.flatnonzero(rec[1:]!=rec[:-1])\n    return float(np.sum((rec[j+1]-rec[j])*prec[j+1]))\n\n\ndef nms(detections, threshold=.5):\n    keep=[]\n    for det in sorted(detections,key=lambda d:-d[\'score\']):\n        boxes=[d[\'box\'] for d in keep if d[\'cls\']==det[\'cls\']]\n        if not boxes or iou(det[\'box\'],boxes).max()<=threshold:keep.append(det)\n    return keep\n\n\ndef counts_to_metrics(tp,fp,gt):\n    fn=gt-tp\n    return {\'GT\':int(gt),\'TP\':int(tp),\'FP\':int(fp),\'FN\':int(fn),\n            \'precision\':100*tp/max(tp+fp,1),\'recall\':100*tp/max(gt,1),\n            \'f1\':200*tp/max(2*tp+fp+fn,1)}\n\n\ndef summarize(records, confidence=.3, overlap=.5, floor=.001, use_nms=False, nms_iou=.5, names=NAMES):\n    gt_count=np.zeros(len(names),dtype=int);tp=np.zeros_like(gt_count);fp=np.zeros_like(gt_count)\n    scored=[[] for _ in names];errors=defaultdict(int);outcomes=[]\n    for record in records:\n        gt=np.asarray(record[\'ground_truth\'],dtype=float).reshape(-1,5)\n        ds=[d for d in record[\'detections\'] if d[\'score\']>=floor]\n        if use_nms:ds=nms(ds,nms_iou)\n        for c in range(len(names)):gt_count[c]+=int((gt[:,0]==c).sum())\n        all_matches,_=match(ds,gt,overlap)\n        for row in all_matches:scored[row[\'cls\']].append([row[\'score\'],row[\'tp\'],row[\'fp\']])\n        op=[d for d in ds if d[\'score\']>=confidence];rows,missing=match(op,gt,overlap)\n        for row in rows:\n            tp[row[\'cls\']]+=row[\'tp\'];fp[row[\'cls\']]+=row[\'fp\']\n            if row[\'fp\']:errors[row[\'category\']]+=1\n        outcomes.append({\'video_id\':record[\'video_id\'],\'frame_idx\':record[\'frame_idx\'],\n                         \'detections\':op,\'ground_truth\':gt.tolist(),\'matching\':rows,\'unmatched_gt_indices\':missing})\n    classes=[]\n    for c,name in enumerate(names):\n        classes.append({\'class\':name,**counts_to_metrics(tp[c],fp[c],gt_count[c]),\'AP50\':100*interpolated_ap(scored[c],gt_count[c])})\n    pooled=counts_to_metrics(int(tp.sum()),int(fp.sum()),int(gt_count.sum()))\n    pooled[\'mAP50\']=float(np.mean([x[\'AP50\'] for x in classes]))\n    pooled.update(confidence=confidence,matching_iou=overlap,ap_score_floor=floor,nms=use_nms,\n                  per_class=classes,fp_categories=dict(errors))\n    return pooled,outcomes\n\n\ndef bootstrap(records, replicates=1000, seed=2026, confidence=.3, overlap=.5, floor=.001):\n    """Cache per-video matches; resample videos with multiplicity, retaining every frame/GT."""\n    groups=defaultdict(list)\n    for r in records:groups[r[\'video_id\']].append(r)\n    cached=[]\n    for video in sorted(groups):\n        rs=groups[video];totals,_=summarize(rs,confidence,overlap,floor);scores=[[] for _ in NAMES]\n        for r in rs:\n            rows,_=match([d for d in r[\'detections\'] if d[\'score\']>=floor],r[\'ground_truth\'],overlap)\n            for row in rows:scores[row[\'cls\']].append([row[\'score\'],row[\'tp\'],row[\'fp\']])\n        cached.append((totals,[np.asarray(v,dtype=float).reshape(-1,3) for v in scores]))\n    if not cached:raise ValueError(\'No test videos\')\n    rng=np.random.default_rng(seed);samples=[]\n    for _ in range(replicates):\n        # Every selection is a distinct replica of the video, including its matched targets.\n        selected=rng.integers(0,len(cached),len(cached));tp=fp=gt=0;aps=[]\n        for j in selected:\n            t=cached[j][0];tp+=t[\'TP\'];fp+=t[\'FP\'];gt+=t[\'GT\']\n        metrics=counts_to_metrics(tp,fp,gt)\n        for c in range(len(NAMES)):\n            chunks=[cached[j][1][c] for j in selected]\n            class_gt=sum(cached[j][0][\'per_class\'][c][\'GT\'] for j in selected)\n            aps.append(100*interpolated_ap(np.concatenate(chunks),class_gt))\n        samples.append([metrics[\'precision\'],metrics[\'recall\'],float(np.mean(aps))])\n    arr=np.asarray(samples)\n    return {\'method\':\'video-cluster percentile bootstrap\',\'replicates\':replicates,\'bootstrap_seed\':seed,\n            \'videos\':len(cached),\'checkpoint_fixed\':True,\'interval_level\':.95,\n            \'percentile_intervals\':{name:np.percentile(arr[:,i],[2.5,97.5]).tolist()\n                                   for i,name in enumerate([\'precision\',\'recall\',\'mAP50\'])},\n            \'replicate_values\':arr.tolist(),\'absent_class_ap\':0,\'class_denominator\':6}\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--predictions\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--bootstrap\',type=int,default=1000);p.add_argument(\'--bootstrap-seed\',type=int,default=2026)\n    p.add_argument(\'--confidence\',type=float,default=.3);p.add_argument(\'--iou\',type=float,default=.5)\n    p.add_argument(\'--ap-floor\',type=float,default=.001);a=p.parse_args()\n    records=json.loads(Path(a.predictions).read_text());out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\n    primary, outcomes=summarize(records,a.confidence,a.iou,a.ap_floor)\n    primary[\'predictions_sha256\']=sha256(a.predictions)\n    write_json(out/\'summary.json\',primary);write_json(out/\'matching_records.json\',outcomes)\n    sweep=[]\n    for threshold in [.1,.2,.3,.4,.5,.6,.7]:\n        item,_=summarize(records,threshold,a.iou,a.ap_floor)\n        sweep.append({k:item[k] for k in [\'confidence\',\'GT\',\'TP\',\'FP\',\'FN\',\'precision\',\'recall\',\'f1\']})\n    write_json(out/\'threshold_sweep.json\',sweep)\n    sensitivity,_=summarize(records,a.confidence,a.iou,a.ap_floor,True,.5)\n    write_json(out/\'nms_sensitivity.json\',sensitivity)\n    if a.bootstrap:write_json(out/\'bootstrap.json\',bootstrap(records,a.bootstrap,a.bootstrap_seed,a.confidence,a.iou,a.ap_floor))\n    print(\'Record-derived analysis exported:\',out)\n\n\nif __name__==\'__main__\':main()\n', 'artifacts/README.md': '# Experimental evidence\n\nNo historical experiment files were present in the supplied notebook. This\nrelease generates them from actual execution or supports depositing your existing\nrecords. Do not substitute reference values or software-test fixtures for them.\n\nFor each run retain `config.yaml`, `config.json`, `environment.json`,\n`dataset_audit.json`, `tensor_shapes.json`, 30-row `history.csv` (with learning\nrate), `best.pt`, `last.pt`, `train.log`, and `evaluate.log`.\n\nEvaluation retains checkpoint-linked `predictions.json` including video/frame\nIDs, GT boxes and scored predictions; unrounded/rounded summaries; class counts/AP;\nmatching records; hashes; environment and shape trace.\n\nReference analyses retain threshold sweep, mutually exclusive FP categories,\nNMS sensitivity, 1,000 video bootstrap replicate metrics and intervals, and logs.\nIndependent seeds and component/context variants retain separate complete records.\n\nProfiling retains individual forward durations, actual stage durations, precision,\ninput shape, synchronization, hardware/software and peak allocated-memory scope.\nFigure 8 exports original frame overlays plus matching/source-hash records.\nAll other experimental charts read real run files, never `reported/` reference\nvalues. Missing analyses are recorded in a figure manifest rather than fabricated.\n\nCheckpoint epoch fields in this release are one based. Older checkpoint fields\nmay be zero based; do not change their interpretation silently. Preserve\nhistorical originals when importing records, and document compatibility changes.\n', 'audit_dataset.py': '"""Fail-fast annotation, video-split, temporal-frame and target collision audit."""\nimport argparse\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom runtime import write_json, sha256\n\nCOLUMNS = [\'video_id\', \'frame_idx\', \'cls\', \'x\', \'y\', \'w\', \'h\', \'t\']\n\n\ndef frame_path(root, video_id, frame_idx):\n    for name in [f\'{frame_idx}.jpg\', f\'{frame_idx:05d}.jpg\']:\n        path = Path(root) / video_id / name\n        if path.is_file(): return path\n    raise FileNotFoundError(f\'Missing temporal frame: {root}/{video_id}/{frame_idx}.jpg\')\n\n\ndef validate_annotations(df, frames_root, num_classes=6):\n    if df.empty or not set(COLUMNS) <= set(df.columns):\n        raise ValueError(\'Nonempty CSV required with columns: \' + \',\'.join(COLUMNS))\n    if df[COLUMNS].isna().any().any(): raise ValueError(\'Missing annotation values\')\n    root = Path(frames_root).resolve()\n    for value in df.video_id.unique():\n        if not isinstance(value, str) or not value or not (root / value).resolve().is_relative_to(root):\n            raise ValueError(f\'Invalid or escaping video_id: {value!r}\')\n    for col in [\'frame_idx\', \'cls\', \'t\']:\n        a = pd.to_numeric(df[col], errors=\'raise\').to_numpy()\n        if not np.isfinite(a).all() or not (a == np.floor(a)).all(): raise ValueError(f\'{col} must be finite integers\')\n    if (df.frame_idx < 0).any(): raise ValueError(\'Frame indices must be zero based and nonnegative\')\n    if not df.cls.isin(range(-1, num_classes)).all() or not df.t.isin([0, 1]).all():\n        raise ValueError(\'Invalid class or temporal metadata\')\n    a = df[[\'x\',\'y\',\'w\',\'h\']].to_numpy(dtype=float)\n    if not np.isfinite(a).all(): raise ValueError(\'Nonfinite bounding boxes\')\n    p = df.cls >= 0; x,y,w,h = a[p].T\n    if ((w <= 0) | (h <= 0) | (x-w/2 < -1e-7) | (y-h/2 < -1e-7) |\n        (x+w/2 > 1+1e-7) | (y+h/2 > 1+1e-7)).any(): raise ValueError(\'Invalid normalized center-format box\')\n    for _, rows in df.groupby([\'video_id\', \'frame_idx\'], sort=False):\n        neg = rows.cls == -1\n        if neg.any() and (len(rows) != 1 or (rows[[\'x\',\'y\',\'w\',\'h\',\'t\']].to_numpy() != 0).any()):\n            raise ValueError(\'Negative frame requires exactly one cls=-1 sentinel with zero coordinates and t=0\')\n    if df.duplicated(COLUMNS).any(): raise ValueError(\'Duplicate annotation rows: resolve rather than silently drop\')\n\n\ndef audit(cfg, check_frames=True, grid=None):\n    result, ids = {}, {}\n    for split in [\'train\', \'val\', \'test\']:\n        path = cfg[\'data\'][split+\'_csv\']; df = pd.read_csv(path)\n        validate_annotations(df, cfg[\'data\'][\'frames_root\'], cfg[\'data\'][\'num_classes\'])\n        ids[split] = set(df.video_id.unique()); groups = df.groupby([\'video_id\',\'frame_idx\'], sort=False)\n        positive = sum((rows.cls >= 0).any() for _,rows in groups)\n        item = {\'annotation_rows\':len(df), \'videos\':len(ids[split]), \'key_frames\':len(groups),\n                \'positive_key_frames\':int(positive), \'negative_key_frames\':len(groups)-int(positive),\n                \'gt_instances\':int((df.cls >= 0).sum()), \'csv_sha256\':sha256(path),\n                \'class_support\':{name:{\'videos\':int(df[df.cls==c].video_id.nunique()),\n                    \'gt_instances\':int((df.cls==c).sum())} for c,name in enumerate(cfg[\'data\'][\'class_names\'])}}\n        if check_frames:\n            checked = set()\n            for (video, frame), rows in groups:\n                for index in range(max(0,int(frame)-cfg[\'data\'][\'clip_len\']+1),int(frame)+1):\n                    key=(video,index)\n                    if key not in checked:\n                        frame_path(cfg[\'data\'][\'frames_root\'],video,index); checked.add(key)\n            item[\'required_frames_checked\'] = len(checked)\n        if grid:\n            height,width = grid; collisions=0\n            for _,rows in groups:\n                occupied=set()\n                for row in rows[rows.cls>=0].itertuples():\n                    cell=(min(height-1,int(row.y*height)),min(width-1,int(row.x*width)))\n                    collisions += cell in occupied; occupied.add(cell)\n            item.update(collision_assignments=int(collisions), collision_grid=list(grid),\n                        collision_percent=100*collisions/max(item[\'gt_instances\'],1))\n        result[split] = item\n    for a,b in [(\'train\',\'val\'),(\'train\',\'test\'),(\'val\',\'test\')]:\n        overlap=ids[a]&ids[b]\n        if overlap: raise ValueError(f\'Video identifier leakage {a}/{b}: {sorted(overlap)}\')\n    result[\'checks\']={\'video_identifier_disjoint\':True, \'all_retained_boxes_valid\':True,\n                      \'frames_checked\':check_frames, \'renamed_content_duplicates_checked\':False}\n    return result\n\n\nif __name__ == \'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--config\',default=\'configs/config.yaml\')\n    p.add_argument(\'--output\',default=\'artifacts/dataset_audit.json\');p.add_argument(\'--grid\',nargs=2,type=int)\n    p.add_argument(\'--skip-frame-check\',action=\'store_true\');a=p.parse_args()\n    cfg=yaml.safe_load(Path(a.config).read_text())\n    write_json(a.output,audit(cfg,not a.skip_frame_check,a.grid))\n    print(\'Dataset audit passed:\',a.output)\n', 'check_reported_values.py': '"""Compare generated records with author-reported manuscript values; never overwrite results."""\nimport argparse\nimport json\nfrom pathlib import Path\nfrom runtime import write_json\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--results\',required=True);p.add_argument(\'--reference\',default=\'reported/reference_values.json\')\n    p.add_argument(\'--output\',default=\'artifacts/reference_comparison.json\');p.add_argument(\'--tolerance\',type=float,default=.0051)\n    a=p.parse_args();actual=json.loads(Path(a.results).read_text());reference=json.loads(Path(a.reference).read_text())\n    checks={}\n    for key,target in reference[\'reference_run\'].items():\n        if key not in actual:continue\n        value=actual[key];checks[key]={\'reported\':target,\'generated\':value,\'difference\':value-target,\n                                      \'matches_rounding\':abs(value-target)<=a.tolerance}\n    write_json(a.output,{\'reference_provenance\':reference[\'provenance\'],\'checks\':checks,\n                         \'all_compared_values_match\':all(v[\'matches_rounding\'] for v in checks.values())})\n    print(a.output)\n\n\nif __name__==\'__main__\':main()\n', 'configs/config.yaml': 'data:\n  train_csv: ucfcrime2local/annotations/train.csv\n  val_csv: ucfcrime2local/annotations/val.csv\n  test_csv: ucfcrime2local/annotations/test.csv\n  frames_root: ucfcrime2local/frames\n  clip_len: 16\n  img_size: 224\n  num_classes: 6\n  class_names:\n  - Arrest\n  - Assault\n  - Burglary\n  - Robbery\n  - Stealing\n  - Vandalism\nmodel:\n  spatial_backbone: yolov8n\n  temporal_backbone: r3d_18\n  inter_channels: 256\n  fusion_channels: 256\n  attn_heads: 4\n  pretrained: true\n  variant: complete\ntrain:\n  optimizer: AdamW\n  learning_rate: 0.0001\n  weight_decay: 0.0005\n  batch_size: 8\n  grad_accum_steps: 2\n  epochs: 30\n  lr_decay_epochs:\n  - 10\n  - 18\n  - 24\n  lr_decay_factor: 0.5\n  lambda_box: 5.0\n  lambda_conf: 1.0\n  amp: true\n  num_workers: 2\n  seed: 42\neval:\n  score_threshold: 0.3\n  iou_threshold: 0.5\n  nms_iou: 0.5\n  ap_score_floor: 0.001\n  nms: false\n  batch_size: 1\nhardware:\n  device: cuda\n', 'configure.py': '"""Resolve dataset paths without modifying the versioned base configuration."""\nimport argparse\nfrom pathlib import Path\nimport yaml\n\n\ndef configure(base, data_root, seed=42, variant=\'complete\', clip_len=16, device=\'cuda\'):\n    cfg=yaml.safe_load(Path(base).read_text());root=Path(data_root).expanduser().resolve()\n    for split in [\'train\',\'val\',\'test\']:cfg[\'data\'][split+\'_csv\']=str(root/\'annotations\'/f\'{split}.csv\')\n    cfg[\'data\'][\'frames_root\']=str(root/\'frames\');cfg[\'data\'][\'clip_len\']=clip_len\n    cfg[\'train\'][\'seed\']=seed;cfg[\'model\'][\'variant\']=variant;cfg[\'hardware\'][\'device\']=device\n    return cfg\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--data-root\',required=True);p.add_argument(\'--base\',default=\'configs/config.yaml\')\n    p.add_argument(\'--output\',default=\'configs/local.yaml\');p.add_argument(\'--device\',default=\'cuda\')\n    a=p.parse_args();out=Path(a.output);out.parent.mkdir(parents=True,exist_ok=True)\n    out.write_text(yaml.safe_dump(configure(a.base,a.data_root,device=a.device),sort_keys=False))\n    print(out)\n', 'dataset.py': '"""Loads consecutive RGB clips ending at a labeled key frame.\nRequires verified normalized center-format bounding boxes and explicit negative frames.\nClass labels on isolated images are insufficient for temporal localization training."""\nimport os\nfrom typing import List, Dict\n\nimport cv2\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\n\nclass AnomalyClipDataset(Dataset):\n    def __init__(self, csv_path: str, frames_root: str, clip_len: int = 16,\n                 img_size: int = 224, num_classes: int = 6, is_train: bool = True):\n        self.frames_root = frames_root\n        self.clip_len = clip_len\n        self.img_size = img_size\n        self.num_classes = num_classes\n        self.is_train = is_train\n\n        from audit_dataset import validate_annotations\n        from pathlib import Path\n        self.frames_root = str(Path(frames_root).resolve())\n        df = pd.read_csv(csv_path)\n        validate_annotations(df, self.frames_root, num_classes)\n        required = {"video_id", "frame_idx", "cls", "x", "y", "w", "h", "t"}\n        missing = required - set(df.columns)\n        if missing:\n            raise ValueError(f"Annotation CSV is missing columns: {missing}")\n\n        # Group boxes by (video_id, frame_idx) -> list of box dicts\n        self.samples: List[Dict] = []\n        grouped = df.groupby(["video_id", "frame_idx"])\n        for (video_id, frame_idx), group in grouped:\n            boxes = group.loc[group["cls"] >= 0, ["cls", "x", "y", "w", "h"]].values.astype(np.float32)\n            self.samples.append({\n                "video_id": str(video_id),\n                "frame_idx": int(frame_idx),\n                "boxes": boxes,  # [num_boxes, 5] -> (cls, x, y, w, h)\n            })\n\n    def __len__(self):\n        return len(self.samples)\n\n    def _load_frame(self, video_id: str, frame_idx: int) -> np.ndarray:\n        path = os.path.join(self.frames_root, video_id, f"{frame_idx}.jpg")\n        if not os.path.isfile(path):\n            path = os.path.join(self.frames_root, video_id, f"{frame_idx:05d}.jpg")\n        img = cv2.imread(path)\n        if img is None:\n            raise FileNotFoundError(f"Missing frame: {path}")\n        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)\n        img = cv2.resize(img, (self.img_size, self.img_size))\n        return img\n\n    def _load_clip(self, video_id: str, key_frame_idx: int) -> torch.Tensor:\n        """\n        Builds X_t = {I_max(0,t-k+1), ..., I_t} ending at the annotated key frame,\n        clamping at the start of the video if fewer than k prior frames exist.\n        """\n        frames = []\n        start = max(0, key_frame_idx - self.clip_len + 1)\n        indices = list(range(start, key_frame_idx + 1))\n        # left-pad by repeating the first available frame if the clip is short\n        while len(indices) < self.clip_len:\n            indices.insert(0, indices[0])\n\n        for idx in indices:\n            frames.append(self._load_frame(video_id, idx))\n\n        clip = np.stack(frames, axis=0)              # [T, H, W, 3]\n        clip = clip.astype(np.float32) / 255.0\n        clip = (clip - 0.45) / 0.225                  # simple normalization\n        clip = torch.from_numpy(clip).permute(3, 0, 1, 2)  # [3, T, H, W]\n        return clip\n\n    def __getitem__(self, idx: int):\n        sample = self.samples[idx]\n        video_id = sample["video_id"]\n        frame_idx = sample["frame_idx"]\n\n        clip = self._load_clip(video_id, frame_idx)         # [3, T, H, W] for temporal branch\n        key_frame = clip[:, -1, :, :]                        # [3, H, W] for spatial branch\n\n        boxes = torch.from_numpy(sample["boxes"])            # [num_boxes, 5]\n\n        return {\n            "clip": clip,\n            "key_frame": key_frame,\n            "boxes": boxes,               # (cls, x, y, w, h), normalized\n            "video_id": video_id,\n            "frame_idx": frame_idx,\n        }\n\n\ndef collate_fn(batch):\n    """Handles variable number of boxes per sample."""\n    clips = torch.stack([b["clip"] for b in batch], dim=0)\n    key_frames = torch.stack([b["key_frame"] for b in batch], dim=0)\n    boxes = [b["boxes"] for b in batch]  # list of [num_boxes_i, 5] tensors\n    video_ids = [b["video_id"] for b in batch]\n    frame_idxs = [b["frame_idx"] for b in batch]\n    return {\n        "clips": clips,\n        "key_frames": key_frames,\n        "boxes": boxes,\n        "video_ids": video_ids,\n        "frame_idxs": frame_idxs,\n    }\n', 'evaluate.py': '"""FP32 evaluation with checkpoint-linked predictions and measured localization summaries."""\nimport argparse\nfrom pathlib import Path\nimport torch\nimport yaml\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\nfrom dataset import AnomalyClipDataset, collate_fn\nfrom metrics import decode_predictions\nfrom analysis import summarize\nfrom audit_dataset import audit\nfrom runtime import resolve_device, load_selected, write_json, sha256, environment, initialize\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--config\',default=\'configs/config.yaml\');p.add_argument(\'--checkpoint\',required=True)\n    p.add_argument(\'--output\',default=\'runs/reference/evaluation\');a=p.parse_args()\n    cfg=yaml.safe_load(Path(a.config).read_text());device=resolve_device(cfg)\n    audit(cfg,check_frames=True)\n    out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\n    model,ckpt=load_selected(cfg,a.checkpoint,device)\n    initialize(model,cfg,device,out/\'tensor_shapes.json\')\n    ds=AnomalyClipDataset(cfg[\'data\'][\'test_csv\'],cfg[\'data\'][\'frames_root\'],cfg[\'data\'][\'clip_len\'],\n                          cfg[\'data\'][\'img_size\'],cfg[\'data\'][\'num_classes\'],False)\n    loader=DataLoader(ds,batch_size=1,shuffle=False,num_workers=cfg[\'train\'][\'num_workers\'],collate_fn=collate_fn)\n    records=[]\n    with torch.inference_mode():\n        for batch in tqdm(loader,desc=\'Test evaluation\'):\n            outputs=model(batch[\'clips\'].to(device),batch[\'key_frames\'].to(device))\n            detections=decode_predictions(outputs[\'cls_logits\'][0],outputs[\'reg\'][0],outputs[\'conf_logits\'][0],cfg[\'eval\'][\'ap_score_floor\'])\n            records.append({\'video_id\':batch[\'video_ids\'][0],\'frame_idx\':int(batch[\'frame_idxs\'][0]),\n                            \'detections\':detections,\'ground_truth\':batch[\'boxes\'][0].tolist()})\n    predictions=out/\'predictions.json\';write_json(predictions,records)\n    summary,outcomes=summarize(records,cfg[\'eval\'][\'score_threshold\'],cfg[\'eval\'][\'iou_threshold\'],cfg[\'eval\'][\'ap_score_floor\'])\n    summary[\'provenance\']={\'checkpoint_sha256\':sha256(a.checkpoint),\'predictions_sha256\':sha256(predictions),\n                           \'test_csv_sha256\':sha256(cfg[\'data\'][\'test_csv\']),\'config_sha256\':sha256(a.config),\n                           \'checkpoint_epoch\':ckpt.get(\'epoch\'),\'seed\':cfg[\'train\'][\'seed\'],\n                           \'variant\':cfg[\'model\'][\'variant\'],\'clip_length\':cfg[\'data\'][\'clip_len\'],\n                           \'matching\':\'highest-IoU same-class; no reassignment\',\'ap\':\'all-point interpolation\',\n                           \'precision\':\'FP32\',\'checkpoint_selection\':\'minimum validation mean mini-batch loss\'}\n    write_json(out/\'results.json\',summary);write_json(out/\'matching_records.json\',outcomes)\n    write_json(out/\'per_class.json\',summary[\'per_class\']);write_json(out/\'environment.json\',environment(device))\n    print({k:summary[k] for k in [\'precision\',\'recall\',\'f1\',\'mAP50\',\'GT\',\'TP\',\'FP\',\'FN\']})\n\n\nif __name__==\'__main__\':main()\n', 'export_overlays.py': '"""Figure 8: render original source frames with their record-derived boxes and outcomes."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import Rectangle\nimport cv2\nfrom analysis import NAMES, summarize\nfrom audit_dataset import frame_path\nfrom runtime import write_json, sha256\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--predictions\',required=True);p.add_argument(\'--frames-root\',required=True)\n    p.add_argument(\'--selection\',help=\'JSON list of {video_id,frame_idx}; without this, select observed outcome categories\')\n    p.add_argument(\'--output\',default=\'runs/figures/Figure8.PNG\');p.add_argument(\'--confidence\',type=float,default=.3)\n    a=p.parse_args();records=json.loads(Path(a.predictions).read_text());_,outcomes=summarize(records,a.confidence)\n    if a.selection:\n        requested=json.loads(Path(a.selection).read_text());index={(r[\'video_id\'],r[\'frame_idx\']):r for r in outcomes}\n        chosen=[index[(r[\'video_id\'],r[\'frame_idx\'])] for r in requested]\n    else:\n        chosen=[];used=set()\n        for category in [\'true_positive\',\'insufficient_overlap\',\'miss\',\'duplicate_or_assignment_conflict\',\'wrong_class\',\'background_or_negligible_overlap\']:\n            for r in outcomes:\n                key=(r[\'video_id\'],r[\'frame_idx\'])\n                exists=bool(r[\'unmatched_gt_indices\']) if category==\'miss\' else any(m[\'category\']==category for m in r[\'matching\'])\n                if key not in used and exists:chosen.append(r);used.add(key);break\n    if not chosen:raise ValueError(\'No matching examples available\')\n    cols=3;rows=(len(chosen)+cols-1)//cols;fig,axes=plt.subplots(rows,cols,figsize=(15,5*rows),squeeze=False)\n    sources=[]\n    for ax,r in zip(axes.flat,chosen):\n        path=frame_path(a.frames_root,r[\'video_id\'],r[\'frame_idx\']);image=cv2.imread(str(path))\n        if image is None:raise ValueError(f\'Unreadable source: {path}\')\n        image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB);height,width=image.shape[:2];ax.imshow(image)\n        for c,x,y,w,h in r[\'ground_truth\']:\n            ax.add_patch(Rectangle(((x-w/2)*width,(y-h/2)*height),w*width,h*height,fill=False,color=\'#338653\',lw=2))\n            ax.text((x-w/2)*width,(y-h/2)*height,\'GT: \'+NAMES[int(c)],color=\'white\',fontsize=8,bbox={\'facecolor\':\'#338653\',\'alpha\':.85})\n        by_index={m[\'detection_index\']:m for m in r[\'matching\']}\n        for i,d in enumerate(r[\'detections\']):\n            x,y,w,h=d[\'box\'];m=by_index[i]\n            ax.add_patch(Rectangle(((x-w/2)*width,(y-h/2)*height),w*width,h*height,fill=False,color=\'#2469B2\',lw=2,ls=\'--\'))\n            label=f"Pred: {NAMES[d[\'cls\']]} {d[\'score\']:.2f}; same-class IoU {m[\'same_class_iou\']:.2f}"\n            ax.text((x-w/2)*width,(y+h/2)*height,label,color=\'white\',fontsize=7,bbox={\'facecolor\':\'#2469B2\',\'alpha\':.85})\n        categories=sorted({m[\'category\'] for m in r[\'matching\']});text=\', \'.join(x.replace(\'_\',\' \') for x in categories)\n        if r[\'unmatched_gt_indices\']:text+=f"; missed GT: {len(r[\'unmatched_gt_indices\'])}"\n        ax.set_title(f"{r[\'video_id\']} | frame {r[\'frame_idx\']}\\n{text}",fontsize=9);ax.axis(\'off\')\n        sources.append({**r,\'source_frame_sha256\':sha256(path)})\n    for ax in list(axes.flat)[len(chosen):]:ax.axis(\'off\')\n    fig.suptitle(\'Ground-truth annotations and reference-run predictions\',fontsize=16)\n    fig.tight_layout(rect=(0,0,1,.95));out=Path(a.output);out.parent.mkdir(parents=True,exist_ok=True)\n    fig.savefig(out,dpi=300,bbox_inches=\'tight\');plt.close(fig)\n    write_json(out.with_suffix(\'.records.json\'),{\'predictions_sha256\':sha256(a.predictions),\'confidence\':a.confidence,\n               \'matching_iou\':.5,\'note\':\'Source pixels are read directly; same-class IoU does not represent overlap with a wrong-class target.\',\'examples\':sources})\n    print(out)\n\n\nif __name__==\'__main__\':main()\n', 'export_release.py': '"""Separate source and real execution artifacts with SHA-256 manifests."""\nimport argparse\nimport zipfile\nfrom pathlib import Path\nfrom runtime import sha256,write_json\n\n\ndef archive(paths,base,target):\n    with zipfile.ZipFile(target,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in paths:z.write(p,p.relative_to(base))\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--repository\',default=\'.\');p.add_argument(\'--runs\',default=\'runs\');p.add_argument(\'--output\',default=\'release\')\n    a=p.parse_args();root=Path(a.repository).resolve();out=Path(a.output).resolve();out.mkdir(parents=True,exist_ok=True)\n    source=[]\n    for path in root.rglob(\'*\'):\n        if not path.is_file():continue\n        relative=path.relative_to(root)\n        if any(x in relative.parts for x in [\'__pycache__\',\'.git\',\'runs\',\'ucfcrime2local\',\'release\',\'.ipynb_checkpoints\']):continue\n        if path.is_relative_to(out):continue\n        if path.suffix in [\'.py\',\'.md\',\'.yaml\',\'.yml\',\'.json\',\'.txt\',\'.cff\',\'.ipynb\'] or path.name==\'.gitignore\':source.append(path)\n    write_json(out/\'source_sha256.json\',{str(p.relative_to(root)):sha256(p) for p in source})\n    archive(source,root,out/\'repository_source.zip\')\n    runs=Path(a.runs).resolve()\n    if runs.exists():\n        actual=[p for p in runs.rglob(\'*\') if p.is_file() and \'__pycache__\' not in p.parts]\n        if actual:\n            write_json(out/\'run_sha256.json\',{str(p.relative_to(runs)):sha256(p) for p in actual})\n            archive(actual,runs,out/\'experimental_records.zip\')\n    print(\'Release exported:\',out)\n\n\nif __name__==\'__main__\':main()\n', 'extract_frames.py': '"""Decode original videos to zero-based JPEG frames required by the annotation manifests."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport cv2\nimport pandas as pd\nfrom runtime import sha256,write_json\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--video-map\',required=True,help=\'JSON {video_id: original video filepath}\')\n    p.add_argument(\'--annotations\',nargs=\'+\',required=True);p.add_argument(\'--frames-root\',required=True)\n    p.add_argument(\'--clip-length\',type=int,default=32,help=\'Use 32 to support all context-length experiments\')\n    p.add_argument(\'--overwrite\',action=\'store_true\');a=p.parse_args()\n    mapping=json.loads(Path(a.video_map).read_text());df=pd.concat([pd.read_csv(x) for x in a.annotations]);root=Path(a.frames_root).resolve();records=[]\n    for video,rows in df.groupby(\'video_id\'):\n        directory=(root/video).resolve()\n        if not directory.is_relative_to(root):raise ValueError(\'video_id escapes frame root\')\n        directory.mkdir(parents=True,exist_ok=True)\n        if video not in mapping:raise KeyError(f\'Missing video mapping: {video}\')\n        required=set()\n        for key in rows.frame_idx.unique():required.update(range(max(0,int(key)-a.clip_length+1),int(key)+1))\n        if not required:continue\n        cap=cv2.VideoCapture(mapping[video]);index=0;written=0\n        if not cap.isOpened():raise ValueError(f\'Cannot open video {video}\')\n        try:\n            while index<=max(required):\n                ok,frame=cap.read()\n                if not ok:raise ValueError(f\'Video {video} ends before required frame {index}\')\n                if index in required:\n                    target=directory/f\'{index:05d}.jpg\'\n                    if target.exists() and not a.overwrite:raise FileExistsError(f\'Frame exists; do not mix extraction versions: {target}\')\n                    if not cv2.imwrite(str(target),frame,[cv2.IMWRITE_JPEG_QUALITY,95]):raise IOError(f\'Cannot write {target}\')\n                    written+=1\n                index+=1\n        finally:cap.release()\n        records.append({\'video_id\':video,\'original_video_sha256\':sha256(mapping[video]),\'frames_written\':written,\n                        \'index_base\':0,\'JPEG_quality\':95,\'color_file_format\':\'BGR encoding; loader converts RGB\'})\n    write_json(root/\'extraction_provenance.json\',records)\n\n\nif __name__==\'__main__\':main()\n', 'import_annotations.py': '"""Explicit CSV adaptation; does not infer native dataset formats or split assignments."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport pandas as pd\nfrom audit_dataset import validate_annotations\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--input\',required=True);p.add_argument(\'--output\',required=True)\n    p.add_argument(\'--frames-root\',required=True);p.add_argument(\'--mapping\',help=\'JSON mapping canonical column name -> input column name\')\n    p.add_argument(\'--coordinates\',choices=[\'normalized-xywh\',\'pixel-xyxy\'],required=True)\n    p.add_argument(\'--frame-base\',type=int,choices=[0,1],required=True);a=p.parse_args()\n    df=pd.read_csv(a.input)\n    if a.mapping:\n        mapping=json.loads(Path(a.mapping).read_text());df=df.rename(columns={value:key for key,value in mapping.items()})\n    df[\'frame_idx\']=df[\'frame_idx\']-a.frame_base\n    if a.coordinates==\'pixel-xyxy\':\n        required={\'xmin\',\'ymin\',\'xmax\',\'ymax\',\'image_width\',\'image_height\'}\n        if not required<=set(df):raise ValueError(\'Pixel conversion requires explicit xmin,ymin,xmax,ymax,image_width,image_height columns\')\n        if (df[[\'image_width\',\'image_height\']]<=0).any().any():raise ValueError(\'Invalid original dimensions\')\n        df[\'x\']=(df.xmin+df.xmax)/(2*df.image_width);df[\'y\']=(df.ymin+df.ymax)/(2*df.image_height)\n        df[\'w\']=(df.xmax-df.xmin)/df.image_width;df[\'h\']=(df.ymax-df.ymin)/df.image_height\n        neg=df.cls==-1;df.loc[neg,[\'x\',\'y\',\'w\',\'h\']]=0\n    validate_annotations(df,a.frames_root,6)\n    from audit_dataset import COLUMNS\n    out=Path(a.output);out.parent.mkdir(parents=True,exist_ok=True);df[COLUMNS].to_csv(out,index=False);print(out)\n\n\nif __name__==\'__main__\':main()\n', 'losses.py': '"""\nLoss functions matching the manuscript\'s "Optimization Objective" subsection:\n\n    L_cls  = -1/N sum [y_i log(y_hat_i) + (1-y_i) log(1 - y_hat_i)]   (BCE)\n    L_box  = 1 - IoU(B, B_hat)\n    L_conf = BCE(S, S_hat)\n    L_total = L_cls + lambda_1 * L_box + lambda_2 * L_conf\n\nLabel assignment: one regression target per cell containing a ground-truth\nbox center. Later boxes overwrite regression coordinates on collisions;\nclassification labels accumulate at that cell. This is not FCOS center sampling.\n"""\nfrom typing import List, Dict\nimport torch\nimport torch.nn.functional as F\n\nfrom utils import box_iou_xywh\n\n\ndef assign_targets(boxes: List[torch.Tensor], grid_h: int, grid_w: int,\n                    num_classes: int, device) -> Dict[str, torch.Tensor]:\n    """\n    boxes: list (len=batch) of [num_boxes_i, 5] tensors (cls, x, y, w, h), normalized [0,1]\n    Returns dense per-cell targets of shape [B, grid_h, grid_w, ...]\n    """\n    B = len(boxes)\n    cls_target = torch.zeros(B, num_classes, grid_h, grid_w, device=device)\n    box_target = torch.zeros(B, 4, grid_h, grid_w, device=device)\n    conf_target = torch.zeros(B, 1, grid_h, grid_w, device=device)\n    pos_mask = torch.zeros(B, grid_h, grid_w, dtype=torch.bool, device=device)\n\n    for b in range(B):\n        gt = boxes[b]\n        if gt.numel() == 0:\n            continue\n        gt = gt.to(device)\n        for row in gt:\n            c, x, y, w, h = row.tolist()\n            c = int(c)\n            # grid cell containing the box center\n            gi = min(int(x * grid_w), grid_w - 1)\n            gj = min(int(y * grid_h), grid_h - 1)\n\n            cls_target[b, c, gj, gi] = 1.0\n            box_target[b, :, gj, gi] = torch.tensor([x, y, w, h], device=device)\n            conf_target[b, 0, gj, gi] = 1.0\n            pos_mask[b, gj, gi] = True\n\n    return {\n        "cls_target": cls_target,\n        "box_target": box_target,\n        "conf_target": conf_target,\n        "pos_mask": pos_mask,\n    }\n\n\ndef hybrid_loss(preds: Dict[str, torch.Tensor], boxes: List[torch.Tensor],\n                 num_classes: int, lambda_box: float = 5.0, lambda_conf: float = 1.0):\n    """\n    preds: dict with \'cls_logits\' [B,C,H,W], \'reg\' [B,4,H,W], \'conf_logits\' [B,1,H,W]\n    boxes: list of [num_boxes_i, 5] ground-truth tensors per sample\n    """\n    device = preds["cls_logits"].device\n    B, C, H, W = preds["cls_logits"].shape\n\n    targets = assign_targets(boxes, H, W, num_classes, device)\n\n    # ---- L_cls : binary cross-entropy over the full classification map ----\n    l_cls = F.binary_cross_entropy_with_logits(\n        preds["cls_logits"], targets["cls_target"], reduction="mean"\n    )\n\n    # ---- L_box : 1 - IoU(B, B_hat), computed only at positive cells ----\n    pos_mask = targets["pos_mask"]  # [B,H,W]\n    num_pos = pos_mask.sum().clamp(min=1)\n\n    pred_box = preds["reg"].sigmoid()  # keep predictions in [0,1] normalized space\n    pred_box_pos = pred_box.permute(0, 2, 3, 1)[pos_mask]     # [num_pos, 4]\n    gt_box_pos = targets["box_target"].permute(0, 2, 3, 1)[pos_mask]  # [num_pos, 4]\n\n    if pred_box_pos.numel() > 0:\n        ious = box_iou_xywh(pred_box_pos, gt_box_pos).diagonal()\n        l_box = (1.0 - ious).sum() / num_pos\n    else:\n        l_box = torch.tensor(0.0, device=device)\n\n    # ---- L_conf : BCE(S, S_hat) over the full confidence map ----\n    l_conf = F.binary_cross_entropy_with_logits(\n        preds["conf_logits"], targets["conf_target"], reduction="mean"\n    )\n\n    l_total = l_cls + lambda_box * l_box + lambda_conf * l_conf\n\n    return {\n        "loss": l_total,\n        "loss_cls": l_cls.detach(),\n        "loss_box": l_box.detach() if torch.is_tensor(l_box) else torch.tensor(l_box),\n        "loss_conf": l_conf.detach(),\n    }\n', 'metrics.py': '"""\nEvaluation metrics matching the manuscript\'s "Evaluation Metrics" subsection:\n\n    Precision = TP / (TP + FP)\n    Recall    = TP / (TP + FN)\n    F1        = 2 * Precision * Recall / (Precision + Recall)\n    mAP       = (1/N) * sum_i AP_i\n\nDetections and ground truths are matched per-image using IoU >= iou_threshold,\nusing the manuscript-specific highest-IoU same-class matcher. Already\nmatched highest-IoU targets are rejected without reassignment. AP uses all-point\ninterpolation, not COCO 101-point AP.\n"""\nfrom typing import List, Dict\nimport numpy as np\nimport torch\n\nfrom utils import box_iou_xywh\n\n\ndef decode_predictions(cls_logits, reg, conf_logits, score_threshold=0.3):\n    """\n    Converts a single sample\'s dense prediction maps into a flat list of\n    (cls, score, x, y, w, h) detections in normalized image coordinates.\n    """\n    C, H, W = cls_logits.shape\n    cls_probs = cls_logits.sigmoid()      # [C,H,W]\n    conf_probs = conf_logits.sigmoid()    # [1,H,W]\n    box = reg.sigmoid()                    # [4,H,W], already (x,y,w,h) normalized\n\n    scores, class_ids = cls_probs.max(dim=0)   # [H,W] each\n    scores = scores * conf_probs[0]             # combine class score with objectness\n\n    dets = []\n    ys, xs = torch.where(scores >= score_threshold)\n    for gy, gx in zip(ys.tolist(), xs.tolist()):\n        dets.append({\n            "grid_y": gy, "grid_x": gx,\n            "cls": int(class_ids[gy, gx].item()),\n            "score": float(scores[gy, gx].item()),\n            "box": box[:, gy, gx].tolist(),   # (x, y, w, h)\n        })\n    return dets\n\n\ndef match_detections_to_gt(dets: List[Dict], gts: np.ndarray, iou_threshold: float = 0.5):\n    """\n    dets: list of {\'cls\',\'score\',\'box\'} for one image, SORTED by score descending\n    gts:  [num_gt, 5] array (cls, x, y, w, h) for the same image\n    Returns per-detection tp/fp flags (1/0) aligned with `dets` order.\n    """\n    matched_gt = set()\n    tp = np.zeros(len(dets))\n    fp = np.zeros(len(dets))\n\n    if len(gts) == 0:\n        fp[:] = 1\n        return tp, fp\n\n    gt_boxes = torch.tensor(gts[:, 1:], dtype=torch.float32)\n    gt_classes = gts[:, 0].astype(int)\n\n    for i, det in enumerate(dets):\n        det_box = torch.tensor([det["box"]], dtype=torch.float32)\n        same_cls_idx = np.where(gt_classes == det["cls"])[0]\n        if len(same_cls_idx) == 0:\n            fp[i] = 1\n            continue\n        ious = box_iou_xywh(det_box, gt_boxes[same_cls_idx]).squeeze(0)\n        best_iou, best_local = ious.max(0)\n        best_idx = same_cls_idx[best_local.item()]\n        if best_iou.item() >= iou_threshold and best_idx not in matched_gt:\n            tp[i] = 1\n            matched_gt.add(best_idx)\n        else:\n            fp[i] = 1\n    return tp, fp\n\n\ndef compute_precision_recall_f1(all_dets: List[List[Dict]], all_gts: List[np.ndarray],\n                                 iou_threshold: float = 0.5):\n    """\n    all_dets: list (len=num_images) of detection lists\n    all_gts:  list (len=num_images) of [num_gt_i, 5] arrays\n    """\n    total_tp, total_fp, total_gt = 0, 0, 0\n    for dets, gts in zip(all_dets, all_gts):\n        dets_sorted = sorted(dets, key=lambda d: -d["score"])\n        tp, fp = match_detections_to_gt(dets_sorted, gts, iou_threshold)\n        total_tp += tp.sum()\n        total_fp += fp.sum()\n        total_gt += len(gts)\n\n    precision = total_tp / max(total_tp + total_fp, 1e-9)\n    recall = total_tp / max(total_gt, 1e-9)\n    f1 = 2 * precision * recall / max(precision + recall, 1e-9)\n    return float(precision), float(recall), float(f1)\n\n\ndef compute_ap_per_class(all_dets: List[List[Dict]], all_gts: List[np.ndarray],\n                          cls_id: int, iou_threshold: float = 0.5) -> float:\n    """All-point interpolated average precision for one class."""\n    scored = []   # (score, tp, fp) per detection, across all images\n    num_gt_total = 0\n\n    for img_idx, (dets, gts) in enumerate(zip(all_dets, all_gts)):\n        cls_dets = [d for d in dets if d["cls"] == cls_id]\n        cls_dets = sorted(cls_dets, key=lambda d: -d["score"])\n        cls_gts = gts[gts[:, 0] == cls_id] if len(gts) > 0 else np.zeros((0, 5))\n        num_gt_total += len(cls_gts)\n\n        tp, fp = match_detections_to_gt(cls_dets, cls_gts, iou_threshold)\n        for d, t, f in zip(cls_dets, tp, fp):\n            scored.append((d["score"], t, f))\n\n    if num_gt_total == 0 or len(scored) == 0:\n        return 0.0\n\n    scored.sort(key=lambda s: -s[0])\n    tp_cum = np.cumsum([s[1] for s in scored])\n    fp_cum = np.cumsum([s[2] for s in scored])\n\n    recalls = tp_cum / max(num_gt_total, 1e-9)\n    precisions = tp_cum / np.maximum(tp_cum + fp_cum, 1e-9)\n\n    # all-point interpolation\n    mrec = np.concatenate(([0.0], recalls, [1.0]))\n    mpre = np.concatenate(([1.0], precisions, [0.0]))\n    for i in range(len(mpre) - 2, -1, -1):\n        mpre[i] = max(mpre[i], mpre[i + 1])\n    idx = np.where(mrec[1:] != mrec[:-1])[0]\n    ap = np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])\n    return float(ap)\n\n\ndef compute_map(all_dets: List[List[Dict]], all_gts: List[np.ndarray],\n                 num_classes: int, iou_threshold: float = 0.5) -> float:\n    """mAP = (1/N) * sum_i AP_i, over all configured classes; absent classes receive zero AP."""\n    aps = []\n    present_classes = set()\n    for gts in all_gts:\n        if len(gts) > 0:\n            present_classes.update(gts[:, 0].astype(int).tolist())\n\n    for c in range(num_classes):\n        aps.append(compute_ap_per_class(all_dets, all_gts, c, iou_threshold))\n\n    return float(np.mean(aps)) if aps else 0.0\n', 'model.py': '"""Hybrid YOLOv8n spatial features and R3D-18 temporal features with attention.\nResearch implementation; not the official YOWOv3 baseline. Initialization errors fail explicitly."""\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\n# ---------------------------------------------------------------------------\n# Spatial branch (YOLO-based, matches phi_s in the manuscript)\n# ---------------------------------------------------------------------------\nclass SpatialBranch(nn.Module):\n    def __init__(self, backbone_name: str = "yolov8n", pretrained: bool = True,\n                 out_channels: int = 256):\n        super().__init__()\n        self.out_channels = out_channels\n        self._use_ultralytics = False\n\n        try:\n            from ultralytics import YOLO\n            yolo = YOLO(f"{backbone_name}.pt" if pretrained else f"{backbone_name}.yaml")\n            # Ultralytics stores the underlying nn.Module in .model; we keep\n            # the backbone+neck layers and drop the Detect() head so we get\n            # raw multi-scale feature maps instead of final predictions.\n            self.pretrained_weight_path = getattr(yolo, "ckpt_path", None) if pretrained else None\n            full_model = yolo.model.model  # nn.Sequential of all layers\n            self.backbone = full_model[:-1]  # drop Detect head (last module)\n            self._use_ultralytics = True\n            # project whatever channel count the last kept layer outputs to\n            # a fixed out_channels so the rest of the pipeline is backbone-agnostic\n            self.channel_proj = nn.LazyConv2d(out_channels, kernel_size=1)\n        except Exception as e:\n            raise RuntimeError("YOLO initialization failed; no architecture fallback is permitted") from e\n\n    def forward(self, key_frame: torch.Tensor) -> torch.Tensor:\n        """key_frame: [B, 3, H, W] -> F_s: [B, out_channels, H\', W\']"""\n        x, saved = key_frame, []\n        for layer in self.backbone:\n            if layer.f != -1:\n                x = saved[layer.f] if isinstance(layer.f, int) else [x if j == -1 else saved[j] for j in layer.f]\n            x = layer(x)\n            saved.append(x)\n        feat = x\n        if isinstance(feat, (list, tuple)):\n            feat = feat[-1]\n        feat = self.channel_proj(feat)\n        return feat\n\n\n# ---------------------------------------------------------------------------\n# Temporal branch (YOWOv3-inspired 3D backbone, matches phi_t)\n# ---------------------------------------------------------------------------\nclass TemporalBranch(nn.Module):\n    def __init__(self, backbone_name: str = "r3d_18", pretrained: bool = True,\n                 out_channels: int = 256):\n        super().__init__()\n        from torchvision.models.video import r3d_18, R3D_18_Weights\n\n        weights = R3D_18_Weights.DEFAULT if pretrained else None\n        net = r3d_18(weights=weights)\n        self.pretrained_weights_url = weights.url if weights is not None else None\n        from pathlib import Path\n        self.pretrained_weight_path = str(Path(torch.hub.get_dir()) / "checkpoints" / weights.url.rsplit("/", 1)[-1]) if weights is not None else None\n        # drop avgpool + fc; keep spatio-temporal conv stack\n        self.backbone = nn.Sequential(\n            net.stem, net.layer1, net.layer2, net.layer3, net.layer4,\n        )\n        self.channel_proj = nn.Conv3d(512, out_channels, kernel_size=1)\n\n    def forward(self, clip: torch.Tensor) -> torch.Tensor:\n        """clip: [B, 3, T, H, W] -> F_t: [B, out_channels, H\', W\'] (temporal dim squeezed)"""\n        feat = self.backbone(clip)             # [B, C, T\', H\', W\']\n        feat = self.channel_proj(feat)\n        feat = feat.mean(dim=2)                # squeeze temporal dim -> [B, C, H\', W\']\n        return feat\n\n\n# ---------------------------------------------------------------------------\n# Multimodal fusion module: F_c = Concat(F_s_hat, F_t_hat); F_fusion = Conv(sigma(F_c))\n# ---------------------------------------------------------------------------\nclass FusionModule(nn.Module):\n    def __init__(self, spatial_channels: int, temporal_channels: int, fusion_channels: int):\n        super().__init__()\n        self.proj_s = nn.Conv2d(spatial_channels, fusion_channels, kernel_size=1)   # P_s\n        self.proj_t = nn.Conv2d(temporal_channels, fusion_channels, kernel_size=1)  # P_t\n        self.fuse_conv = nn.Sequential(\n            nn.Conv2d(fusion_channels * 2, fusion_channels, kernel_size=3, padding=1),\n            nn.BatchNorm2d(fusion_channels),\n            nn.SiLU(inplace=True),   # sigma(.)\n        )\n\n    def forward(self, f_s: torch.Tensor, f_t: torch.Tensor) -> torch.Tensor:\n        f_s_hat = self.proj_s(f_s)\n        f_t_hat = self.proj_t(f_t)\n        # align spatial resolution (temporal branch may output a coarser map)\n        if f_t_hat.shape[-2:] != f_s_hat.shape[-2:]:\n            f_t_hat = F.interpolate(f_t_hat, size=f_s_hat.shape[-2:],\n                                     mode="bilinear", align_corners=False)\n        f_c = torch.cat([f_s_hat, f_t_hat], dim=1)   # Concat(F_s_hat, F_t_hat)\n        f_fusion = self.fuse_conv(f_c)\n        return f_fusion\n\n\n# ---------------------------------------------------------------------------\n# Attention-based refinement: A(F) = Softmax(QK^T/sqrt(d)) V ; F_r = F_fusion + A(F_fusion)\n# ---------------------------------------------------------------------------\nclass SelfAttentionRefine(nn.Module):\n    def __init__(self, channels: int, num_heads: int = 4):\n        super().__init__()\n        assert channels % num_heads == 0, "channels must be divisible by num_heads"\n        self.channels = channels\n        self.num_heads = num_heads\n        self.head_dim = channels // num_heads\n\n        self.q_proj = nn.Conv2d(channels, channels, kernel_size=1)\n        self.k_proj = nn.Conv2d(channels, channels, kernel_size=1)\n        self.v_proj = nn.Conv2d(channels, channels, kernel_size=1)\n        self.out_proj = nn.Conv2d(channels, channels, kernel_size=1)\n\n    def forward(self, f_fusion: torch.Tensor) -> torch.Tensor:\n        B, C, H, W = f_fusion.shape\n        N = H * W\n\n        q = self.q_proj(f_fusion).view(B, self.num_heads, self.head_dim, N)\n        k = self.k_proj(f_fusion).view(B, self.num_heads, self.head_dim, N)\n        v = self.v_proj(f_fusion).view(B, self.num_heads, self.head_dim, N)\n\n        attn = torch.einsum("bhdn,bhdm->bhnm", q, k) / (self.head_dim ** 0.5)  # QK^T/sqrt(d)\n        attn = attn.softmax(dim=-1)                                            # Softmax(.)\n        out = torch.einsum("bhnm,bhdm->bhdn", attn, v)                         # (.) V\n        out = out.reshape(B, C, H, W)\n        out = self.out_proj(out)\n\n        f_r = f_fusion + out    # F_r = F_fusion + A(F_fusion)\n        return f_r\n\n\n# ---------------------------------------------------------------------------\n# Anomaly detection head: C = sigma(W_c F_r + b_c) ; B = W_b F_r + b_b\n# ---------------------------------------------------------------------------\nclass DetectionHead(nn.Module):\n    def __init__(self, in_channels: int, num_classes: int):\n        super().__init__()\n        self.cls_head = nn.Conv2d(in_channels, num_classes, kernel_size=1)  # W_c, b_c\n        self.reg_head = nn.Conv2d(in_channels, 4, kernel_size=1)             # W_b, b_b\n        self.conf_head = nn.Conv2d(in_channels, 1, kernel_size=1)            # confidence S\n\n    def forward(self, f_r: torch.Tensor):\n        cls_logits = self.cls_head(f_r)          # [B, num_classes, H, W] (pre-sigmoid)\n        reg = self.reg_head(f_r)                  # [B, 4, H, W]  (cx, cy, w, h) offsets\n        conf_logits = self.conf_head(f_r)         # [B, 1, H, W]  (pre-sigmoid)\n        return cls_logits, reg, conf_logits\n\n\n# ---------------------------------------------------------------------------\n# Full hybrid framework\n# ---------------------------------------------------------------------------\nclass HybridYOLOYOWOv3(nn.Module):\n    def __init__(self, num_classes: int, spatial_backbone: str = "yolov8n",\n                 temporal_backbone: str = "r3d_18", inter_channels: int = 256,\n                 fusion_channels: int = 256, attn_heads: int = 4, pretrained: bool = True, variant: str = "complete"):\n        super().__init__()\n        if variant not in {"complete", "fusion_no_attention", "spatial_only", "temporal_only"}:\n            raise ValueError(f"Unknown variant: {variant}")\n        self.variant = variant\n        self.spatial_branch = SpatialBranch(spatial_backbone, pretrained, inter_channels) if variant != "temporal_only" else None\n        self.temporal_branch = TemporalBranch(temporal_backbone, pretrained, inter_channels) if variant != "spatial_only" else None\n        self.fusion = FusionModule(inter_channels, inter_channels, fusion_channels) if variant in {"complete", "fusion_no_attention"} else None\n        self.single_projection = nn.Conv2d(inter_channels, fusion_channels, 1) if self.fusion is None else None\n        self.attn_refine = SelfAttentionRefine(fusion_channels, attn_heads) if variant == "complete" else nn.Identity()\n        self.detect_head = DetectionHead(fusion_channels, num_classes)\n\n    def forward(self, clip: torch.Tensor, key_frame: torch.Tensor):\n        if self.variant == "spatial_only":\n            f_fusion = self.single_projection(self.spatial_branch(key_frame))\n        elif self.variant == "temporal_only":\n            f_fusion = self.single_projection(self.temporal_branch(clip))\n        else:\n            f_s = self.spatial_branch(key_frame)\n            f_t = self.temporal_branch(clip)\n            f_fusion = self.fusion(f_s, f_t)\n        f_r = self.attn_refine(f_fusion)            # F_r\n        cls_logits, reg, conf_logits = self.detect_head(f_r)\n        return {\n            "cls_logits": cls_logits,    # -> C\n            "reg": reg,                  # -> B\n            "conf_logits": conf_logits,  # -> S\n        }\n\n\ndef build_model(cfg, pretrained=None):\n    m = cfg["model"]\n    return HybridYOLOYOWOv3(num_classes=cfg["data"]["num_classes"],\n        spatial_backbone=m["spatial_backbone"], temporal_backbone=m["temporal_backbone"],\n        inter_channels=m["inter_channels"], fusion_channels=m["fusion_channels"],\n        attn_heads=m["attn_heads"], pretrained=m["pretrained"] if pretrained is None else pretrained,\n        variant=m.get("variant", "complete"))\n', 'plot_results.py': '"""Generate manuscript charts only from executed histories, predictions and profiling records."""\nimport argparse\nimport json\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom runtime import write_json\n\nCOLORS=[\'#1769D2\',\'#009DA6\',\'#7751C9\']\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--runs\',default=\'runs\');p.add_argument(\'--output\',default=\'runs/figures\');a=p.parse_args()\n    root=Path(a.runs);out=Path(a.output);out.mkdir(parents=True,exist_ok=True);status={}\n    ref=root/\'complete_T16_seed42\'\n    plt.rcParams.update({\'font.family\':\'DejaVu Sans\',\'font.size\':11,\'axes.spines.top\':False,\'axes.spines.right\':False})\n    def save(number,fig):\n        fig.tight_layout();fig.savefig(out/f\'Figure{number}.PNG\',dpi=300,bbox_inches=\'tight\');plt.close(fig);status[f\'Figure{number}\']=\'generated from execution records\'\n    def bars(number,title,labels,values,ylabel=\'Score (%)\'):\n        fig,ax=plt.subplots(figsize=(10,6));patch=ax.bar(labels,values,color=[COLORS[i%3] for i in range(len(values))]);ax.set_ylim(0,100)\n        ax.bar_label(patch,fmt=\'%.2f\');ax.set_title(title);ax.set_ylabel(ylabel);save(number,fig)\n    hist=ref/\'history.csv\'\n    if hist.exists():\n        h=pd.read_csv(hist);fig,axes=plt.subplots(2,1,figsize=(10,7),sharex=True)\n        for col,label,color in [(\'train_loss\',\'Training\',COLORS[0]),(\'val_loss\',\'Validation\',COLORS[1])]:axes[0].plot(h.epoch,h[col],label=label,color=color)\n        best=h.loc[h.val_loss.idxmin()];axes[0].axvline(best.epoch,ls=\'--\',color=COLORS[2],label=f\'Validation-selected epoch {int(best.epoch)}\')\n        axes[0].set_ylabel(\'Mean mini-batch loss\');axes[0].legend();axes[1].step(h.epoch,h.learning_rate,where=\'post\');axes[1].set_ylabel(\'Learning rate\');axes[1].set_xlabel(\'Epoch\');save(5,fig)\n    result=ref/\'evaluation/results.json\'\n    if result.exists():\n        r=json.loads(result.read_text());bars(6,\'Aggregate key-frame localization: reference run\',[\'Precision\',\'Recall\',\'mAP@0.5\'],[r[\'precision\'],r[\'recall\'],r[\'mAP50\']])\n        bars(11,\'Class-wise AP at IoU 0.5\',[x[\'class\'] for x in r[\'per_class\']],[x[\'AP50\'] for x in r[\'per_class\']])\n        errors=r[\'fp_categories\'];fig,ax=plt.subplots(figsize=(11,6));labels=list(errors);values=[errors[x] for x in labels]\n        ax.barh([x.replace(\'_\',\' \') for x in labels],values,color=COLORS[0]);ax.set_xlabel(\'False-positive count\');ax.set_title(f\'Reference-run false-positive counts (total {sum(values)})\');save(12,fig)\n    sweep=ref/\'analysis/threshold_sweep.json\'\n    if sweep.exists():\n        rs=json.loads(sweep.read_text());fig,ax=plt.subplots(figsize=(10,6))\n        for key,color in zip([\'precision\',\'recall\',\'f1\'],COLORS):ax.plot([x[\'confidence\'] for x in rs],[x[key] for x in rs],marker=\'o\',label=key,color=color)\n        ax.set(xlabel=\'Confidence threshold\',ylabel=\'Score (%)\',ylim=(0,100),title=\'Fixed-checkpoint threshold sweep\');ax.legend();save(7,fig)\n    profile=ref/\'profiling/profiling.json\'\n    if profile.exists():\n        pr=json.loads(profile.read_text());s=pr[\'offline_serial\'];fig,ax=plt.subplots(figsize=(12,6))\n        ax.barh(s[\'stage_names\'],s[\'stage_mean_ms\'],color=COLORS[0]);ax.set_xlabel(\'Mean duration (ms)\')\n        ax.set_title(\'Measured serial JPEG-frame pipeline latency\')\n        fig.text(.5,.01,f"Total: {s[\'total_mean_ms\']:.2f} ms; {s[\'key_frames_per_second\']:.2f} key frames/s. Live acquisition and queueing excluded.",ha=\'center\',fontsize=9)\n        save(9,fig)\n    variant_labels=[\'spatial_only\',\'temporal_only\',\'fusion_no_attention\',\'complete\'];variant_results=[]\n    for v in variant_labels:\n        file=root/f\'{v}_T16_seed42/evaluation/results.json\'\n        if file.exists():variant_results.append(json.loads(file.read_text()))\n    if len(variant_results)==4:bars(10,\'Component comparison results\',[v.replace(\'_\',\'\\n\') for v in variant_labels],[r[\'mAP50\'] for r in variant_results],\'mAP@0.5 (%)\')\n    seeds=[]\n    for seed in [21,42,84]:\n        file=root/f\'complete_T16_seed{seed}/evaluation/results.json\'\n        if file.exists():seeds.append((seed,json.loads(file.read_text())))\n    if len(seeds)==3:\n        fig,axes=plt.subplots(1,3,figsize=(12,5));stats={}\n        for ax,key,color in zip(axes,[\'precision\',\'recall\',\'mAP50\'],COLORS):\n            values=[r[key] for _,r in seeds];ax.plot([s for s,_ in seeds],values,marker=\'o\',color=color);ax.axhline(np.mean(values),ls=\'--\',color=color)\n            stats[key]={\'mean\':float(np.mean(values)),\'sample_sd\':float(np.std(values,ddof=1))}\n            ax.set(xlabel=\'Initialization seed\',ylabel=\'Score (%)\',title=f\'{key}: SD={np.std(values,ddof=1):.2f} pp\',ylim=(0,100))\n        save(13,fig);write_json(out/\'seed_statistics.json\',stats)\n    clip_results=[]\n    for length in [8,16,32]:\n        file=root/f\'complete_T{length}_seed42/evaluation/results.json\'\n        if file.exists():clip_results.append((length,json.loads(file.read_text())))\n    if len(clip_results)==3:bars(14,\'Retrained clip-length comparisons\',[str(x) for x,_ in clip_results],[r[\'mAP50\'] for _,r in clip_results],\'mAP@0.5 (%)\')\n    boot=ref/\'analysis/bootstrap.json\'\n    if boot.exists() and result.exists():\n        b=json.loads(boot.read_text());r=json.loads(result.read_text());fig,ax=plt.subplots(figsize=(10,5))\n        for i,(key,color) in enumerate(zip([\'precision\',\'recall\',\'mAP50\'],COLORS)):\n            lo,hi=b[\'percentile_intervals\'][key];ax.hlines(i,lo,hi,color=color,lw=3);ax.plot(r[key],i,\'o\',color=color)\n        ax.set_yticks(range(3),[\'Precision\',\'Recall\',\'mAP@0.5\']);ax.set(xlabel=\'Score (%)\',xlim=(0,100),title=\'Video-cluster bootstrap: percentile 95% intervals\');save(15,fig)\n    for number in [5,6,7,9,10,11,12,13,14,15]:\n        status.setdefault(f\'Figure{number}\',\'not generated: required execution records unavailable\')\n    write_json(out/\'figure_manifest.json\',status);print(status)\n\n\nif __name__==\'__main__\':main()\n', 'profile.py': '"""Synchronized forward profiling and serial JPEG-frame pipeline timing; no live-video claim."""\nimport argparse\nimport copy\nimport json\nimport time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport yaml\nfrom runtime import resolve_device, load_selected, initialize, environment, write_json, sha256\nfrom audit_dataset import frame_path\nfrom dataset import AnomalyClipDataset, collate_fn\nfrom metrics import decode_predictions\nfrom losses import hybrid_loss\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--config\',required=True);p.add_argument(\'--checkpoint\',required=True)\n    p.add_argument(\'--output\',default=\'runs/reference/profiling\');p.add_argument(\'--iterations\',type=int,default=100)\n    p.add_argument(\'--warmup\',type=int,default=10);a=p.parse_args()\n    if a.iterations<1 or a.warmup<0:raise ValueError(\'Invalid iteration counts\')\n    cfg=yaml.safe_load(Path(a.config).read_text());device=resolve_device(cfg);out=Path(a.output)\n    model,_=load_selected(cfg,a.checkpoint,device);initialize(model,cfg,device,out/\'tensor_shapes.json\')\n    size=cfg[\'data\'][\'img_size\'];length=cfg[\'data\'][\'clip_len\']\n    clip=torch.randn(1,3,length,size,size,device=device);frame=torch.randn(1,3,size,size,device=device)\n    def sync():\n        if device.type==\'cuda\':torch.cuda.synchronize(device)\n    latencies=[]\n    with torch.inference_mode():\n        for _ in range(a.warmup):model(clip,frame)\n        sync()\n        if device.type==\'cuda\':torch.cuda.reset_peak_memory_stats(device)\n        for _ in range(a.iterations):\n            sync();start=time.perf_counter();model(clip,frame);sync()\n            latencies.append(1000*(time.perf_counter()-start))\n    memory=torch.cuda.max_memory_allocated(device)/1e9 if device.type==\'cuda\' else None\n    mean=float(np.mean(latencies));parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)\n    ops=None;failure=None;counted=None\n    try:\n        from thop import profile\n        counted=copy.deepcopy(model)\n        macs,_=profile(counted,inputs=(clip,frame),verbose=False);ops=2*float(macs)/1e9\n    except Exception as exc:failure=str(exc)\n    finally:\n        if counted is not None:del counted\n    result={\'environment\':environment(device),\'checkpoint_sha256\':sha256(a.checkpoint),\n        \'trainable_parameters\':parameters,\'params_M\':parameters/1e6,\'THOP_accounted_GFLOPs\':ops,\n        \'THOP_failure\':failure,\'THOP_complete_operation_count\':False,\n        \'THOP_limitations\':\'Custom attention einsum QK/AV operations lack dedicated counting rules.\',\n        \'forward\':{\'precision\':\'FP32\',\'batch_size\':1,\'resident_synthetic_inputs\':True,\'warmup\':a.warmup,\n          \'iterations\':a.iterations,\'latencies_ms\':latencies,\'mean_ms\':mean,\'key_frames_per_second\':1000/mean,\n          \'peak_allocated_device_GB\':memory,\'input_shape\':[1,3,length,size,size]},\n        \'excluded\':[\'camera acquisition\',\'frame waiting\',\'networking\',\'display\',\'queueing\',\'asynchronous overlap\']}\n    # Offline timing mirrors the actual dataset loader but exposes separate stage boundaries.\n    import cv2\n    ds=AnomalyClipDataset(cfg[\'data\'][\'test_csv\'],cfg[\'data\'][\'frames_root\'],length,size,6,False)\n    serial=[]; sample_ids=[]\n    with torch.inference_mode():\n        for sample in ds.samples[:a.iterations]:\n            sync();start=time.perf_counter()\n            indices=[max(0,sample[\'frame_idx\']-length+1+k) for k in range(length)]\n            raw=[]\n            for index in indices:\n                image=cv2.imread(str(frame_path(cfg[\'data\'][\'frames_root\'],sample[\'video_id\'],index)))\n                if image is None:raise ValueError(\'Unreadable JPEG frame\')\n                raw.append(image)\n            read_done=time.perf_counter()\n            processed=[cv2.resize(cv2.cvtColor(image,cv2.COLOR_BGR2RGB),(size,size)) for image in raw]\n            arr=(np.stack(processed).astype(np.float32)/255.-.45)/.225\n            cpu_clip=torch.from_numpy(arr).permute(3,0,1,2).unsqueeze(0)\n            cpu_frame=cpu_clip[:,:,-1];prep_done=time.perf_counter()\n            gpu_clip=cpu_clip.to(device);gpu_frame=cpu_frame.to(device);sync();transfer_done=time.perf_counter()\n            outputs=model(gpu_clip,gpu_frame);sync();forward_done=time.perf_counter()\n            dets=decode_predictions(outputs[\'cls_logits\'][0],outputs[\'reg\'][0],outputs[\'conf_logits\'][0],cfg[\'eval\'][\'ap_score_floor\'])\n            json.dumps(dets);sync();end=time.perf_counter()\n            serial.append([1000*(read_done-start),1000*(prep_done-read_done),1000*(transfer_done-prep_done),\n                           1000*(forward_done-transfer_done),1000*(end-forward_done)])\n            sample_ids.append({\'video_id\':sample[\'video_id\'],\'frame_idx\':sample[\'frame_idx\']})\n    durations=np.asarray(serial);serial_mean=float(durations.sum(1).mean())\n    result[\'offline_serial\']={\'input_source\':\'pre-extracted JPEG frame files; not compressed-video container decoding\',\n      \'cache_state\':\'OS cache uncontrolled; potentially warm\',\'samples\':len(serial),\'sample_ids\':sample_ids,\n      \'stage_names\':[\'JPEG read/decode and clip assembly\',\'resize/RGB/normalization\',\'host-to-device transfer\',\n                     \'model forward\',\'prediction decode and in-memory JSON serialization\'],\n      \'per_sample_stage_ms\':serial,\'stage_mean_ms\':durations.mean(0).tolist(),\n      \'total_mean_ms\':serial_mean,\'key_frames_per_second\':1000/serial_mean,\n      \'output_disk_writing_included\':False,\'forward_stage_separate_from_resident_synthetic_profile\':True}\n    del clip,frame,gpu_clip,gpu_frame,outputs\n    # Training scope uses actual annotation-backed clips and accumulation over two batches.\n    if device.type==\'cuda\':\n        torch.cuda.empty_cache();train_ds=AnomalyClipDataset(cfg[\'data\'][\'train_csv\'],cfg[\'data\'][\'frames_root\'],length,size,6,True)\n        batch_size=cfg[\'train\'][\'batch_size\'];accum=cfg[\'train\'][\'grad_accum_steps\']\n        if len(train_ds)<batch_size*accum:raise ValueError(\'Not enough real samples for the declared training memory scope\')\n        model.train();optimizer=torch.optim.AdamW(model.parameters(),lr=cfg[\'train\'][\'learning_rate\'],weight_decay=cfg[\'train\'][\'weight_decay\'])\n        scaler=torch.cuda.amp.GradScaler(enabled=cfg[\'train\'][\'amp\']);optimizer.zero_grad(set_to_none=True)\n        torch.cuda.reset_peak_memory_stats(device)\n        for batch_index in range(accum):\n            batch=collate_fn([train_ds[batch_index*batch_size+k] for k in range(batch_size)])\n            x=batch[\'clips\'].to(device);key=batch[\'key_frames\'].to(device)\n            with torch.cuda.amp.autocast(enabled=cfg[\'train\'][\'amp\']):\n                predictions=model(x,key)\n                loss=hybrid_loss(predictions,batch[\'boxes\'],6,cfg[\'train\'][\'lambda_box\'],cfg[\'train\'][\'lambda_conf\'])[\'loss\']/accum\n            scaler.scale(loss).backward()\n            del predictions,loss,x,key,batch\n        scaler.step(optimizer);scaler.update();sync()\n        result[\'training_memory\']={\'peak_allocated_device_GB\':torch.cuda.max_memory_allocated(device)/1e9,\n          \'batch_size\':batch_size,\'accumulation\':accum,\'amp\':cfg[\'train\'][\'amp\'],\'real_training_clips\':True,\n          \'scope\':\'two accumulated batches and first AdamW step; includes resident model, gradients and optimizer state\'}\n    else:result[\'training_memory\']=None\n    write_json(out/\'profiling.json\',result);print(\'Profiling saved:\',out/\'profiling.json\')\n\n\nif __name__==\'__main__\':main()\n', 'reported/reference_values.json': '{\n  "provenance": "Author-reported manuscript values confirmed by the author; retained as a comparison reference, not output generated by this repository preparation.",\n  "reference_run": {"precision":54.21,"recall":57.85,"f1":55.97,"mAP50":53.44,"GT":10000,"TP":5785,"FP":4887,"FN":4215},\n  "component_mAP50": {"spatial_only":45.0,"temporal_only":41.0,"fusion_no_attention":49.0,"complete":53.44},\n  "clip_length_mAP50": {"8":51.20,"16":53.44,"32":53.70},\n  "class_AP50": {"Arrest":58.20,"Assault":49.80,"Burglary":55.60,"Robbery":52.40,"Stealing":48.90,"Vandalism":55.74},\n  "runs": [{"seed":21,"precision":53.83,"recall":57.31,"mAP50":52.98},{"seed":42,"precision":54.21,"recall":57.85,"mAP50":53.44},{"seed":84,"precision":54.59,"recall":58.39,"mAP50":53.90}],\n  "sample_SD_percentage_points": {"precision":0.38,"recall":0.54,"mAP50":0.46},\n  "percentile_95_intervals": {"precision":[51.10,57.35],"recall":[54.65,61.02],"mAP50":[50.12,56.71]},\n  "bootstrap": {"videos":31,"replicates":1000,"historical_resampling_seed":"not supplied; new execution defaults to 2026 and records it"},\n  "class_aware_NMS_mAP50":54.10,\n  "FP_counts": {"duplicate_or_assignment_conflict":2000,"wrong_class":900,"insufficient_overlap":687,"background_or_negligible_overlap":1300},\n  "training": {"epochs":30,"selected_epoch":27,"selected_validation_loss":1.090,"final_training_loss":0.790,"final_validation_loss":1.110},\n  "profiling": {"params_M":36.00,"THOP_accounted_GFLOPs":80.00,"forward_mean_ms":25.00,"forward_key_frames_per_second":40.00,"inference_peak_allocated_GB":2.40,"training_peak_allocated_GB":9.80,"offline_serial_mean_ms":40.00,"offline_serial_key_frames_per_second":25.00},\n  "partitions": {"train":{"videos":52,"key_frames":20000,"GT_instances":25000},"val":{"videos":13,"key_frames":5000,"GT_instances":6250},"test":{"videos":31,"key_frames":8000,"GT_instances":10000}},\n  "listed_annotation_paths": {"train_pool":31687,"test":13226},\n  "scope_note":"Listed annotation-path counts are not assumed to equal eligible CSV rows, key frames or box instances. Profiling of JPEG files in this release does not establish compressed-video decoding latency. No historical checkpoints, prediction records or logs were supplied in the uploaded notebook."\n}\n', 'requirements.txt': '# Install the matching PyTorch CUDA build separately (see README).\ntorch==2.2.2\ntorchvision==0.17.2\nultralytics==8.2.0\nthop==0.1.1.post2209072238\nnumpy==1.26.4\npandas==2.2.2\nopencv-python==4.10.0.84\nPyYAML==6.0.2\ntqdm==4.66.5\nmatplotlib==3.9.2\nnbformat==5.10.4\n', 'run_experiments.py': '"""Execute manuscript experiment matrix with isolated run directories and raw logs."""\nimport argparse\nimport subprocess\nimport sys\nfrom pathlib import Path\nimport yaml\nfrom configure import configure\nfrom runtime import write_json\n\n\ndef logged(command, path):\n    with Path(path).open(\'w\',encoding=\'utf-8\') as log:\n        proc=subprocess.Popen([sys.executable,\'-u\',*command],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)\n        for line in proc.stdout:print(line,end=\'\');log.write(line);log.flush()\n        code=proc.wait()\n    if code:raise RuntimeError(f\'Command failed; inspect {path}\')\n\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--data-root\',required=True);p.add_argument(\'--output\',default=\'runs\')\n    p.add_argument(\'--all\',action=\'store_true\',help=\'Three seeds, component ablations, 8/32-frame retraining, reference analyses\')\n    p.add_argument(\'--device\',default=\'cuda\');p.add_argument(\'--bootstrap\',type=int,default=1000)\n    p.add_argument(\'--skip-training\',action=\'store_true\',help=\'Analyze already populated run directories\')\n    a=p.parse_args();jobs=[(\'complete\',16,42)]\n    if a.all:jobs += [(\'complete\',16,21),(\'complete\',16,84),(\'spatial_only\',16,42),(\'temporal_only\',16,42),\n                     (\'fusion_no_attention\',16,42),(\'complete\',8,42),(\'complete\',32,42)]\n    summaries=[]\n    for variant,length,seed in jobs:\n        run=Path(a.output)/f\'{variant}_T{length}_seed{seed}\';run.mkdir(parents=True,exist_ok=True)\n        config=run/\'config.yaml\';cfg=configure(\'configs/config.yaml\',a.data_root,seed,variant,length,a.device)\n        if a.skip_training:\n            if not config.exists() or not (run/\'best.pt\').exists():raise FileNotFoundError(f\'Missing existing config/checkpoint in {run}\')\n        else:\n            if (run/\'history.csv\').exists():raise FileExistsError(f\'Run exists: {run}; use --skip-training or a different output root\')\n            config.write_text(yaml.safe_dump(cfg,sort_keys=False))\n            logged([\'train.py\',\'--config\',str(config),\'--out_dir\',str(run)],run/\'train.log\')\n        logged([\'evaluate.py\',\'--config\',str(config),\'--checkpoint\',str(run/\'best.pt\'),\'--output\',str(run/\'evaluation\')],run/\'evaluate.log\')\n        import json\n        summaries.append(json.loads((run/\'evaluation/results.json\').read_text()))\n    write_json(Path(a.output)/\'experiment_results.json\',summaries)\n    ref=Path(a.output)/\'complete_T16_seed42\'\n    logged([\'analysis.py\',\'--predictions\',str(ref/\'evaluation/predictions.json\'),\'--output\',str(ref/\'analysis\'),\n            \'--bootstrap\',str(a.bootstrap)],ref/\'analysis.log\')\n    logged([\'profile.py\',\'--config\',str(ref/\'config.yaml\'),\'--checkpoint\',str(ref/\'best.pt\'),\n            \'--output\',str(ref/\'profiling\')],ref/\'profile.log\')\n    logged([\'plot_results.py\',\'--runs\',a.output,\'--output\',str(Path(a.output)/\'figures\')],Path(a.output)/\'figures.log\')\n\n\nif __name__==\'__main__\':main()\n', 'runtime.py': '"""Shared device selection, provenance, JSON and model initialization."""\nimport hashlib\nimport json\nimport platform\nimport sys\nfrom pathlib import Path\nfrom importlib.metadata import version, PackageNotFoundError\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)\n    p.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef resolve_device(cfg):\n    import torch\n    device = torch.device(cfg[\'hardware\'][\'device\'])\n    if device.type == \'cuda\' and not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA was requested but is unavailable. For a CPU smoke test explicitly set hardware.device: cpu.\')\n    return device\n\n\ndef environment(device=None):\n    import torch\n    packages = {}\n    for name in [\'torch\', \'torchvision\', \'ultralytics\', \'thop\', \'numpy\', \'pandas\', \'opencv-python\', \'PyYAML\']:\n        try: packages[name] = version(name)\n        except PackageNotFoundError: packages[name] = None\n    return {\'python\': sys.version, \'platform\': platform.platform(), \'packages\': packages,\n            \'torch_cuda_build\': torch.version.cuda, \'device\': str(device),\n            \'cudnn_version\': torch.backends.cudnn.version(),\n            \'gpu\': torch.cuda.get_device_name(device) if device and device.type == \'cuda\' else None}\n\n\ndef initialize(model, cfg, device, trace_path=None):\n    import torch\n    size = cfg[\'data\'][\'img_size\']; length = cfg[\'data\'][\'clip_len\']\n    shapes, handles = {}, []\n    def capture(name):\n        def hook(module, inputs, output):\n            if isinstance(output, torch.Tensor): shapes[name] = list(output.shape)\n            elif isinstance(output, dict): shapes[name] = {k: list(v.shape) for k, v in output.items()}\n        return hook\n    for name, module in model.named_modules():\n        if name and any(name == prefix or name == prefix + \'.channel_proj\' or name == prefix + \'.backbone\'\n                        for prefix in [\'spatial_branch\', \'temporal_branch\', \'fusion\', \'attn_refine\', \'detect_head\']):\n            handles.append(module.register_forward_hook(capture(name)))\n    model.eval()\n    with torch.no_grad():\n        outputs = model(torch.zeros(1, 3, length, size, size, device=device),\n                        torch.zeros(1, 3, size, size, device=device))\n    for handle in handles: handle.remove()\n    shapes[\'outputs\'] = {k: list(v.shape) for k, v in outputs.items()}\n    shapes[\'inputs\'] = {\'clip\': [1, 3, length, size, size], \'key_frame\': [1, 3, size, size]}\n    initialization = {}\n    for name in [\'spatial_branch\', \'temporal_branch\']:\n        branch = getattr(model, name, None)\n        path = getattr(branch, \'pretrained_weight_path\', None)\n        initialization[name] = {\'pretrained_weight_path\': str(path) if path else None,\n                                \'sha256\': sha256(path) if path and Path(path).is_file() else None,\n                                \'upstream_url\': getattr(branch, \'pretrained_weights_url\', None)}\n    shapes[\'pretrained_initialization\'] = initialization\n    if trace_path: write_json(trace_path, shapes)\n    return shapes\n\n\ndef load_selected(cfg, checkpoint, device):\n    import torch\n    from model import build_model\n    ckpt = torch.load(checkpoint, map_location=device, weights_only=True)\n    saved = ckpt.get(\'config\')\n    if saved:\n        for section, keys in [(\'model\', [\'variant\', \'spatial_backbone\', \'temporal_backbone\', \'inter_channels\', \'fusion_channels\', \'attn_heads\']),\n                              (\'data\', [\'clip_len\', \'img_size\', \'num_classes\', \'class_names\'])]:\n            for key in keys:\n                if saved[section].get(key) != cfg[section].get(key):\n                    raise ValueError(f\'Checkpoint/config mismatch: {section}.{key}\')\n    model = build_model(cfg, pretrained=False).to(device)\n    initialize(model, cfg, device)\n    model.load_state_dict(ckpt[\'model_state\'], strict=True)\n    return model.eval(), ckpt\n', 'tests/test_model.py': '"""Optional dependency-backed smoke checks; no pretrained downloads or research measurements."""\nimport importlib.util\nimport sys\nimport unittest\nfrom pathlib import Path\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nREADY=all(importlib.util.find_spec(x) is not None for x in [\'torch\',\'torchvision\',\'ultralytics\'])\n\n\n@unittest.skipUnless(READY,\'PyTorch/torchvision/Ultralytics not installed\')\nclass ModelTests(unittest.TestCase):\n    def test_variants_and_empty_ground_truth_loss(self):\n        import torch\n        from model import HybridYOLOYOWOv3\n        from losses import hybrid_loss\n        torch.set_num_threads(2)\n        for variant in [\'complete\',\'fusion_no_attention\',\'spatial_only\',\'temporal_only\']:\n            model=HybridYOLOYOWOv3(6,pretrained=False,variant=variant).eval()\n            with torch.no_grad():out=model(torch.zeros(1,3,8,64,64),torch.zeros(1,3,64,64))\n            self.assertEqual(out[\'cls_logits\'].shape[1],6)\n            loss=hybrid_loss(out,[torch.empty(0,5)],6)\n            self.assertTrue(torch.isfinite(loss[\'loss\']).item())\n            self.assertEqual(loss[\'loss_box\'].item(),0.)\n            del model\n\n\nif __name__==\'__main__\':unittest.main()\n', 'tests/test_protocol.py': "import unittest\nimport tempfile\nfrom pathlib import Path\nimport sys\nimport pandas as pd\nimport numpy as np\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]))\nfrom analysis import match,summarize,bootstrap,nms\nfrom audit_dataset import validate_annotations,audit\n\n\ndef det(c=0,score=.9,box=None):return {'cls':c,'score':score,'box':box or [.5,.5,.4,.4]}\n\n\nclass ProtocolTests(unittest.TestCase):\n    def test_duplicate_wrong_class_and_negative_frames(self):\n        gt=[[0,.5,.5,.4,.4]]\n        rows,missing=match([det(),det(score=.8),det(1,.7)],gt)\n        self.assertEqual([x['category'] for x in rows],['true_positive','duplicate_or_assignment_conflict','wrong_class'])\n        self.assertEqual(missing,[])\n        result,_=summarize([{'video_id':'a','frame_idx':0,'detections':[det(),det(score=.8),det(1,.7)],'ground_truth':gt},\n                            {'video_id':'b','frame_idx':0,'detections':[det()],'ground_truth':[]}])\n        self.assertEqual((result['TP'],result['FP'],result['FN']),(1,3,0))\n        self.assertAlmostEqual(result['precision'],25.)\n        self.assertAlmostEqual(result['mAP50'],100/6)\n    def test_highest_iou_target_is_not_reassigned(self):\n        gt=[[0,.5,.5,.4,.4],[0,.55,.5,.4,.4]]\n        rows,missing=match([det(),det(score=.8)],gt)\n        self.assertEqual(sum(r['tp'] for r in rows),1)\n        self.assertEqual(missing,[1])\n    def test_confidence_is_separate_from_ap_floor(self):\n        r={'video_id':'a','frame_idx':0,'detections':[det(score=.2)],'ground_truth':[[0,.5,.5,.4,.4]]}\n        s,_=summarize([r]);self.assertEqual(s['TP'],0);self.assertEqual(s['FN'],1)\n        self.assertAlmostEqual(s['mAP50'],100/6)\n    def test_nms_class_aware(self):\n        self.assertEqual(len(nms([det(),det(score=.8),det(1,.7)])),2)\n    def test_bootstrap_multiplicity_matches_explicit_replication(self):\n        records=[{'video_id':'a','frame_idx':0,'detections':[det()],'ground_truth':[[0,.5,.5,.4,.4]]},\n                 {'video_id':'b','frame_idx':0,'detections':[],'ground_truth':[[0,.5,.5,.4,.4]]}]\n        seed=7;n=12;actual=bootstrap(records,n,seed);rng=np.random.default_rng(seed)\n        expected=[]\n        for _ in range(n):\n            selected=rng.integers(0,2,2);s,_=summarize([records[int(i)] for i in selected]);expected.append([s['precision'],s['recall'],s['mAP50']])\n        np.testing.assert_allclose(actual['replicate_values'],expected)\n    def test_annotation_sentinel_and_bounds(self):\n        columns=['video_id','frame_idx','cls','x','y','w','h','t']\n        valid=pd.DataFrame([['a',0,0,.5,.5,.4,.4,1],['a',1,-1,0,0,0,0,0]],columns=columns)\n        validate_annotations(valid,'/tmp/frames')\n        invalid=valid.copy();invalid.loc[0,'w']=2\n        with self.assertRaises(ValueError):validate_annotations(invalid,'/tmp/frames')\n        mixed=pd.concat([valid,valid.iloc[[0]].assign(frame_idx=1)])\n        with self.assertRaises(ValueError):validate_annotations(mixed,'/tmp/frames')\n    def test_video_split_leakage(self):\n        with tempfile.TemporaryDirectory() as temp:\n            root=Path(temp);csv=root/'shared.csv'\n            pd.DataFrame([['same',0,-1,0,0,0,0,0]],columns=['video_id','frame_idx','cls','x','y','w','h','t']).to_csv(csv,index=False)\n            cfg={'data':{'frames_root':str(root),'num_classes':6,'class_names':['a','b','c','d','e','f'],\n                         **{s+'_csv':str(csv) for s in ['train','val','test']}}}\n            with self.assertRaises(ValueError):audit(cfg,False)\n\n\nif __name__=='__main__':unittest.main()\n", 'train.py': '"""Train the hybrid model with AdamW and select checkpoints by validation loss."""\nimport argparse\nimport csv\nimport json\nimport os\nimport time\n\nimport torch\nimport yaml\nfrom torch.optim import AdamW\nfrom torch.utils.data import DataLoader\nfrom tqdm import tqdm\n\nfrom dataset import AnomalyClipDataset, collate_fn\nfrom model import build_model\nfrom losses import hybrid_loss\nfrom utils import set_seed, save_checkpoint, count_parameters\n\n\ndef build_dataloader(cfg, split="train"):\n    csv_key = {"train": "train_csv", "val": "val_csv", "test": "test_csv"}[split]\n    ds = AnomalyClipDataset(\n        csv_path=cfg["data"][csv_key],\n        frames_root=cfg["data"]["frames_root"],\n        clip_len=cfg["data"]["clip_len"],\n        img_size=cfg["data"]["img_size"],\n        num_classes=cfg["data"]["num_classes"],\n        is_train=(split == "train"),\n    )\n    return DataLoader(\n        ds,\n        batch_size=cfg["train"]["batch_size"],\n        shuffle=(split == "train"),\n        num_workers=cfg["train"]["num_workers"],\n        collate_fn=collate_fn,\n        drop_last=False,\n    )\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", type=str, default="configs/config.yaml")\n    parser.add_argument("--out_dir", type=str, default="runs/exp1")\n    args = parser.parse_args()\n\n    with open(args.config) as f:\n        cfg = yaml.safe_load(f)\n\n    set_seed(cfg["train"]["seed"])\n    from runtime import resolve_device, environment, write_json\n    from audit_dataset import audit\n    device = resolve_device(cfg)\n    dataset_audit = audit(cfg, check_frames=True)\n    if os.path.exists(os.path.join(args.out_dir, "history.csv")):\n        raise FileExistsError("Existing run history: choose a new output directory to avoid overwriting records.")\n    print(f"[train] using device: {device}")\n\n    # ---- Step 1: annotated dataset ----\n    train_loader = build_dataloader(cfg, "train")\n    val_loader = build_dataloader(cfg, "val")\n\n    # ---- Model ----\n    model = build_model(cfg).to(device)\n\n    from runtime import initialize\n    os.makedirs(args.out_dir, exist_ok=True)\n    shapes = initialize(model, cfg, device, os.path.join(args.out_dir, "tensor_shapes.json"))\n    collision_audit = audit(cfg, check_frames=False, grid=shapes["outputs"]["cls_logits"][-2:])\n    for split in ["train", "val", "test"]:\n        for key in ["collision_assignments", "collision_grid", "collision_percent"]:\n            dataset_audit[split][key] = collision_audit[split][key]\n    print(f"[train] model parameters: {count_parameters(model):.2f} M")\n\n    optimizer = AdamW(model.parameters(), lr=cfg["train"]["learning_rate"],\n                       weight_decay=cfg["train"]["weight_decay"])\n    scheduler = torch.optim.lr_scheduler.MultiStepLR(\n        optimizer, milestones=cfg["train"]["lr_decay_epochs"], gamma=cfg["train"]["lr_decay_factor"]\n    )\n    scaler = torch.cuda.amp.GradScaler(enabled=cfg["train"]["amp"] and device.type == "cuda")\n\n    os.makedirs(args.out_dir, exist_ok=True)\n    best_val_loss = float("inf")\n    with open(os.path.join(args.out_dir,"config.json"),"w") as f: json.dump(cfg,f,indent=2)\n    write_json(os.path.join(args.out_dir, "environment.json"), environment(device))\n    write_json(os.path.join(args.out_dir, "dataset_audit.json"), dataset_audit)\n    history = []\n\n    for epoch in range(cfg["train"]["epochs"]):\n        epoch_lr = optimizer.param_groups[0]["lr"]\n        model.train()\n        running_loss = 0.0\n        optimizer.zero_grad()\n\n        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg[\'train\'][\'epochs\']}")\n        for step, batch in enumerate(pbar):\n            clips = batch["clips"].to(device)\n            key_frames = batch["key_frames"].to(device)\n            boxes = batch["boxes"]  # list, kept on CPU->moved inside loss fn\n\n            # ---- Steps 3-6: forward through spatial+temporal+fusion+head ----\n            with torch.cuda.amp.autocast(enabled=cfg["train"]["amp"] and device.type == "cuda"):\n                preds = model(clips, key_frames)\n                # ---- Step 7: compute losses ----\n                loss_dict = hybrid_loss(\n                    preds, boxes, cfg["data"]["num_classes"],\n                    lambda_box=cfg["train"]["lambda_box"],\n                    lambda_conf=cfg["train"]["lambda_conf"],\n                )\n                group_start = (step // cfg["train"]["grad_accum_steps"]) * cfg["train"]["grad_accum_steps"]\n                group_size = min(cfg["train"]["grad_accum_steps"], len(train_loader) - group_start)\n                loss = loss_dict["loss"] / group_size\n\n            # ---- Step 8: backprop / update ----\n            scaler.scale(loss).backward()\n            if (step + 1) % cfg["train"]["grad_accum_steps"] == 0 or step + 1 == len(train_loader):\n                scaler.step(optimizer)\n                scaler.update()\n                optimizer.zero_grad()\n\n            running_loss += loss_dict["loss"].item()\n            pbar.set_postfix({\n                "loss": f"{loss_dict[\'loss\'].item():.4f}",\n                "cls": f"{loss_dict[\'loss_cls\'].item():.4f}",\n                "box": f"{loss_dict[\'loss_box\'].item():.4f}",\n                "conf": f"{loss_dict[\'loss_conf\'].item():.4f}",\n            })\n\n        scheduler.step()\n        avg_train_loss = running_loss / max(len(train_loader), 1)\n\n        # ---- validation ----\n        model.eval()\n        val_loss = 0.0\n        with torch.no_grad():\n            for batch in val_loader:\n                clips = batch["clips"].to(device)\n                key_frames = batch["key_frames"].to(device)\n                boxes = batch["boxes"]\n                preds = model(clips, key_frames)\n                loss_dict = hybrid_loss(preds, boxes, cfg["data"]["num_classes"],\n                                         lambda_box=cfg["train"]["lambda_box"],\n                                         lambda_conf=cfg["train"]["lambda_conf"])\n                val_loss += loss_dict["loss"].item()\n        avg_val_loss = val_loss / max(len(val_loader), 1)\n\n        print(f"[epoch {epoch+1}] train_loss={avg_train_loss:.4f} val_loss={avg_val_loss:.4f}")\n\n        history.append({"epoch":epoch+1,"train_loss":avg_train_loss,"val_loss":avg_val_loss,"learning_rate":epoch_lr})\n        with open(os.path.join(args.out_dir,"history.csv"),"w",newline="") as f:\n            writer=csv.DictWriter(f,fieldnames=list(history[0]));writer.writeheader();writer.writerows(history)\n        ckpt = {\n            "epoch": epoch + 1,\n            "epoch_index_zero_based": epoch,\n            "model_state": model.state_dict(),\n            "optimizer_state": optimizer.state_dict(),\n            "scheduler_state": scheduler.state_dict(),\n            "scaler_state": scaler.state_dict(),\n            "best_val_loss": min(best_val_loss, avg_val_loss),\n            "config": cfg,\n        }\n        save_checkpoint(ckpt, os.path.join(args.out_dir, "last.pt"))\n        if avg_val_loss < best_val_loss:\n            best_val_loss = avg_val_loss\n            save_checkpoint(ckpt, os.path.join(args.out_dir, "best.pt"))\n            print(f"[epoch {epoch+1}] new best checkpoint saved (val_loss={best_val_loss:.4f})")\n\n    print("[train] Completed fixed 30-epoch schedule; best.pt selected by minimum validation loss.")\n\n\nif __name__ == "__main__":\n    main()\n', 'ucfcrime2local/annotations/SCHEMA.txt': 'Required CSV columns: video_id,frame_idx,cls,x,y,w,h,t\nPlace actual eligible train.csv, val.csv and test.csv here.\nNo annotation rows or negative examples are fabricated.\nSee DATASET.md for index, coordinate, class and sentinel conventions.\n', 'utils.py': '"""\nShared utilities: box IoU, reproducibility, checkpoint I/O.\n"""\nimport os\nimport random\nimport numpy as np\nimport torch\n\n\ndef set_seed(seed: int = 42):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n\n\ndef box_iou_xywh(box1: torch.Tensor, box2: torch.Tensor) -> torch.Tensor:\n    """\n    Vectorized IoU between two sets of boxes in (cx, cy, w, h) format.\n    box1: [N, 4], box2: [M, 4] -> returns [N, M] IoU matrix.\n    """\n    def to_xyxy(b):\n        cx, cy, w, h = b.unbind(-1)\n        return torch.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], dim=-1)\n\n    b1 = to_xyxy(box1)  # [N,4]\n    b2 = to_xyxy(box2)  # [M,4]\n\n    area1 = (b1[:, 2] - b1[:, 0]).clamp(min=0) * (b1[:, 3] - b1[:, 1]).clamp(min=0)\n    area2 = (b2[:, 2] - b2[:, 0]).clamp(min=0) * (b2[:, 3] - b2[:, 1]).clamp(min=0)\n\n    lt = torch.max(b1[:, None, :2], b2[None, :, :2])   # [N,M,2]\n    rb = torch.min(b1[:, None, 2:], b2[None, :, 2:])   # [N,M,2]\n    wh = (rb - lt).clamp(min=0)\n    inter = wh[..., 0] * wh[..., 1]\n\n    union = area1[:, None] + area2[None, :] - inter\n    return inter / union.clamp(min=1e-7)\n\n\ndef save_checkpoint(state: dict, path: str):\n    os.makedirs(os.path.dirname(path), exist_ok=True)\n    torch.save(state, path)\n\n\ndef load_checkpoint(path: str, map_location="cpu") -> dict:\n    return torch.load(path, map_location=map_location)\n\n\ndef count_parameters(model: torch.nn.Module) -> float:\n    """Returns parameter count in millions, matching the \'Params (M)\' column in Table 2/6."""\n    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6\n'}
for relative, content in BUNDLED_FILES.items():
    target = WORK / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
print("Materialized", len(BUNDLED_FILES), "source/documentation files")


## 2. Install the documented environment
Recommended Python 3.10.13; install the matching CUDA 12.1 PyTorch build first if
needed, as documented in README. The pip call below installs pinned requirements.
First pretrained initialization may download upstream weights.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(WORK / "requirements.txt")], check=True)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))


## 3. Point to `ucfcrime2local`
Default: `ucfcrime2local/` beside this notebook. Set `UCF_DATA_ROOT` to a different
absolute path. Required: `frames/<video_id>/<zero_based_index>.jpg` (five-digit
filenames also supported) and actual `annotations/train.csv`, `val.csv`, `test.csv`.
Read DATASET.md for explicit conversion/extraction tools. Do not infer negative
frames or historical train/validation splits from missing annotations.


In [ ]:
import yaml
from configure import configure
DATA_ROOT = Path(os.environ.get("UCF_DATA_ROOT", "ucfcrime2local")).expanduser().resolve()
DEVICE = os.environ.get("UCF_DEVICE", "cuda")
cfg = configure(WORK / "configs/config.yaml", DATA_ROOT, device=DEVICE)
CONFIG = WORK / "configs/local.yaml"
CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
RUN = WORK / "runs/complete_T16_seed42"
RUN.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATA_ROOT)
print("Configuration:", CONFIG)


## 4. Audit actual partitions and temporal context
Fails on leakage, malformed boxes, ambiguous negative sentinels, duplicate rows,
or missing preceding frames. Counts distinguish annotation rows, key frames and
GT instances. Listed annotation-path counts are a separate historical unit.


In [ ]:
from audit_dataset import audit
from runtime import write_json
DATASET_AUDIT = audit(cfg, check_frames=True)
write_json(RUN / "dataset_audit.json", DATASET_AUDIT)
print(json.dumps(DATASET_AUDIT, indent=2))


## 5. Initialize model and record full module shape traces
The projection is initialized before parameter counting or optimizer construction.
Grid dimensions are taken from a real forward, not assumed.


In [ ]:
import torch
from runtime import resolve_device, initialize, environment
from model import build_model
from utils import set_seed
set_seed(cfg["train"]["seed"])
device = resolve_device(cfg)
model = build_model(cfg).to(device)
SHAPES = initialize(model, cfg, device, RUN / "tensor_shapes.json")
write_json(RUN / "environment.json", environment(device))
grid = SHAPES["outputs"]["cls_logits"][-2:]
write_json(RUN / "dataset_audit.json", audit(cfg, check_frames=True, grid=grid))
print(json.dumps(SHAPES, indent=2))
del model
if device.type == "cuda":
    torch.cuda.empty_cache()


## 6. Train the 30-epoch reference run
Captures raw console output. Existing training history is protected from overwrite.
The complete loss history selects `best.pt`; `last.pt` is retained separately.
No test data select the checkpoint. Epoch fields in this release are one based.


In [ ]:
from run_experiments import logged
import contextlib
@contextlib.contextmanager
def repository_directory():
    previous = Path.cwd()
    os.chdir(WORK)
    try:
        yield
    finally:
        os.chdir(previous)
with repository_directory():
    logged(["train.py", "--config", str(CONFIG), "--out_dir", str(RUN)], RUN / "train.log")
import pandas as pd
history = pd.read_csv(RUN / "history.csv")
assert len(history) == 30
best = history.loc[history.val_loss.idxmin()]
print("Selected epoch:", int(best.epoch), "validation loss:", float(best.val_loss))


## 7. Export real predictions, class counts/AP and matching records
FP32 test inference. Every record includes source video/frame IDs, annotations,
normalized predicted boxes, classes and confidence. Results include hashes and
checkpoint provenance, not only the three aggregate percentages.


In [ ]:
RESULTS = RUN / "evaluation"
with repository_directory():
    logged(["evaluate.py", "--config", str(CONFIG), "--checkpoint", str(RUN / "best.pt"),
            "--output", str(RESULTS)], RUN / "evaluate.log")
print((RESULTS / "results.json").read_text())


## 8. Threshold, NMS, false-positive and video-bootstrap analyses
Primary confidence remains 0.3. The sweep is descriptive; it does not tune the test
operating point. Bootstrap uses 31 videos only if those 31 are present in actual
records. The default resampling seed 2026 is newly documented, not a claim about
the unavailable historical bootstrap seed.


In [ ]:
with repository_directory():
    logged(["analysis.py", "--predictions", str(RESULTS / "predictions.json"), "--output", str(RUN / "analysis"),
            "--bootstrap", "1000", "--bootstrap-seed", "2026"], RUN / "analysis.log")
    logged(["check_reported_values.py", "--results", str(RESULTS / "results.json"),
            "--output", str(RUN / "reference_comparison.json")], RUN / "reference_comparison.log")


## 9. Measured profiling
Forward-only: resident synthetic FP32 tensors, batch one, ten warm-ups, 100 timed
iterations with synchronization. THOP omits custom attention matrix operations.
Offline stage timing reads pre-extracted JPEG frames; it does not establish
compressed-video-container decoding latency. Actual training clips are used for
peak allocated training-memory measurement. See README for precise boundaries.


In [ ]:
with repository_directory():
    logged(["profile.py", "--config", str(CONFIG), "--checkpoint", str(RUN / "best.pt"),
            "--output", str(RUN / "profiling")], RUN / "profile.log")


## 10. Independent seeds, components and temporal-context comparisons
Optional full experiment matrix: eight independently trained runs. Set the flag
below to True on your GPU. Existing reference training is preserved: this cell
trains only additional runs. Each variant uses its own validation-selected model.
The proposed single-path ablation implementation is specified in model.py; verify
it matches historical variants before treating new executions as reproductions.


In [ ]:
RUN_ADDITIONAL_EXPERIMENTS = False
if RUN_ADDITIONAL_EXPERIMENTS:
    from configure import configure
    jobs = [("complete",16,21),("complete",16,84),("spatial_only",16,42),
            ("temporal_only",16,42),("fusion_no_attention",16,42),("complete",8,42),("complete",32,42)]
    for variant, length, seed in jobs:
        run = WORK / "runs" / f"{variant}_T{length}_seed{seed}"
        run.mkdir(parents=True, exist_ok=True)
        config = run / "config.yaml"
        config.write_text(yaml.safe_dump(configure(WORK / "configs/config.yaml", DATA_ROOT, seed, variant, length, DEVICE), sort_keys=False))
        with repository_directory():
            logged(["train.py", "--config", str(config), "--out_dir", str(run)], run / "train.log")
            logged(["evaluate.py", "--config", str(config), "--checkpoint", str(run / "best.pt"),
                    "--output", str(run / "evaluation")], run / "evaluate.log")
else:
    print("Additional training not requested; repeated-run and ablation figures need their actual records.")


## 11. Generate figures from execution records
Training convergence is Figure 5; aggregate localization is Figure 6. Figure 8
uses actual original pixels and corresponding prediction/annotation records.
Automatic selection includes only observed cases; use `--selection` with specific
video/frame IDs to select the manuscript's exact examples. Missing variant/run
records are declared in the figure manifest rather than replaced with reference
values. These charts use full 0–100% axes.


In [ ]:
FIGURES = WORK / "runs/figures"
with repository_directory():
    logged(["plot_results.py", "--runs", str(WORK / "runs"), "--output", str(FIGURES)], WORK / "runs/figures.log")
    logged(["export_overlays.py", "--predictions", str(RESULTS / "predictions.json"),
            "--frames-root", cfg["data"]["frames_root"], "--output", str(FIGURES / "Figure8.PNG")], RUN / "overlays.log")


## 12. Export source and real experiment archives
The dataset is uploaded separately. The export hashes actual files and does not
fabricate missing evidence. Upload this notebook with the source archive; attach
real run archives, original manifests and dataset provenance to the Zenodo deposit.
Add the actual code license, repository URL and minted DOI before public release.


In [ ]:
with repository_directory():
    logged(["export_release.py", "--repository", str(WORK), "--runs", str(WORK / "runs"),
            "--output", str(WORK / "release")], WORK / "export.log")
print("Source archive:", WORK / "release/repository_source.zip")
print("Executed run archive:", WORK / "release/experimental_records.zip")
print("Upload the original dataset separately with its applicable terms and attribution.")
